# Block Influence in Deep Residual Networks — Companion Notebook

**Computational (coding-based) project — *Artificial and Biological Neural Systems* (supervisor: Prof. U. Hasson)**
Author: **Andrea Tatini**.

This notebook is the **Google Colab submission** accompanying the report
*"Block Influence in Deep Residual Networks: Silent Representational Degradation,
Block Interaction, and a Null Result for BI-Weighted Knowledge Distillation"*.
Per the course guidelines, it **contains the code that produces every figure and
table reported in the paper**.

---

## What this notebook does

| Section | Content | Report ref. | Runtime (T4 GPU) |
|---|---|---|---|
| 1 | CIFAR-10 + frozen ResNet-50 teacher (94.64 % test acc.), calibration set, intact reference features $F_{intact}$ | §3.1–3.2 | ~4 min |
| 2 | The three Block-Influence metrics for all 16 Bottleneck blocks: **BIgeo**, **BIacc**, **BIrep** (+ **BIrep\_gram**, **BIrep\_class**) | §2.4, §5.1 | ~20 min |
| 3 | Metric agreement (Kendall's $\tau$, Jaccard@{3,5}), silent-failure analysis of **layer4.0** (entropy/Wilcoxon, per-class CKA, class-pair mergers), simulated pruning | §5.1, E1–E2 | ~10 min |
| 4 | **Real progressive pruning**: k = 1…16 blocks ablated simultaneously, 3 orderings → superadditivity ~2.2× at k = 9 | §5.2, E3 | ~25 min |
| 5 | Phase 4: weighted SP-KD — live diagnosis of the two mechanical causes of the null result; figures/tables of the null result from archived artifacts; full re-training code behind a flag | §5.3–5.6, E4 | ~3 min (+ training if enabled) |

All generated artifacts are saved to `/content/results` and `/content/figures`
(a zip for download is produced at the end).

## How to run

`Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Default flags reproduce everything except Phase-4 re-training
(12 × 200-epoch runs ≈ 72 GPU-hours — not feasible in one Colab session; see §5).

## Reproducibility notes

* Seed 42 fixes `random` / `numpy` / `torch`; cuDNN runs in deterministic mode
  for all measurement phases (`set_seed`, ported from `utils.py`).
* The calibration set (2500 train images, 250/class) is rebuilt deterministically
  with `numpy.random.default_rng(42)` — identical indices to the original runs.
* The teacher checkpoint is downloaded from Hugging Face
  [`edadaltocg/resnet50_cifar10`](https://huggingface.co/edadaltocg/resnet50_cifar10)
  (the same weights used in the report; sanity-checked against 0.9464).
* Phase-4 training results are loaded from an embedded snapshot of the original
  run artifacts (`ARCHIVED_PHASE4`, extracted from `phase4_results/*.json`);
  the notebook also contains complete from-scratch training + post-hoc code so
  every number can be regenerated given enough GPU time.
* Minor documented deviations from the repository pipeline: (i) BIrep,
  BIrep\_gram and BIrep\_class are computed from a **single** ablated feature
  extraction per block (the repo used two identical passes); (ii) the auxiliary
  multi-layer CKA propagation profile (§6.2 "known blemish") is omitted — no
  figure in the paper depends on it.


## Contents

1. [Setup: environment, configuration, execution flags](#s0)
2. [Data & teacher model (Phase 1)](#s1)
3. [Block Influence metrics (Phase 2)](#s2)
4. [Comparative analysis & the layer4.0 silent failure (Phase 3)](#s3)
5. [Real progressive pruning (Phase 3, E3)](#s4)
6. [Phase 4 — BI-weighted SP-KD: the null result](#s5)
7. [Artifact map & export](#s6)


<a id="s0"></a>
## 0 · Setup — environment, configuration, execution flags

Everything the pipeline needs is defined once here (port of the repo's
`config.py` + module-level imports). The single most load-bearing objects are:
the **ablation context manager** (`ablated_block`) and the **centred linear CKA**
(`linear_cka`) — both shown in §2/§3 below.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")  # cuBLAS determinism guard (set before torch initialises CUDA)

import hashlib, json, math, random, re, tarfile, time, urllib.request
from contextlib import contextmanager, ExitStack
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset
from torchvision import datasets, transforms
from torchvision.models.resnet import Bottleneck, BasicBlock, ResNet

from scipy import stats
from tqdm.auto import tqdm
from IPython.display import display

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT        = Path("/content")
DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
CKPT_DIR    = ROOT / "checkpoints"
PHASE4_DIR  = ROOT / "phase4_results"
for d in (DATA_DIR, RESULTS_DIR, FIGURES_DIR, CKPT_DIR, PHASE4_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42

# ── Data constants ─────────────────────────────────────────────────────────
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)
N_CALIBRATION      = 2500     # 250 per class × 10 classes, drawn from the TRAIN split
SAMPLES_PER_CLASS  = 250
BATCH_SIZE_INFERENCE = 256    # test-set passes (BIacc)
BATCH_SIZE_CALIB     = 256    # calibration-set passes (BIgeo / BIrep)
NUM_WORKERS          = 2

BASELINE_ACCURACY_EXPECTED = 0.9464   # report §5.1
ACCURACY_FAIL_THRESHOLD    = 0.90     # hard sanity-check floor

# ── Block registry (canonical order shared by all phases) ──────────────────
TARGET_BLOCKS = [
    "layer1.0", "layer1.1", "layer1.2",
    "layer2.0", "layer2.1", "layer2.2", "layer2.3",
    "layer3.0", "layer3.1", "layer3.2", "layer3.3", "layer3.4", "layer3.5",
    "layer4.0", "layer4.1", "layer4.2",
]
DOWNSAMPLING_BLOCKS = {"layer1.0", "layer2.0", "layer3.0", "layer4.0"}
STAGES = ["layer1", "layer2", "layer3", "layer4"]
STAGE_BLOCKS = {
    "layer1": ["layer1.0", "layer1.1", "layer1.2"],
    "layer2": ["layer2.0", "layer2.1", "layer2.2", "layer2.3"],
    "layer3": ["layer3.0", "layer3.1", "layer3.2", "layer3.3", "layer3.4", "layer3.5"],
    "layer4": ["layer4.0", "layer4.1", "layer4.2"],
}

CKA_EPSILON = 1e-8
ALPHA = 0.05
SECONDARY_DELTA_THRESHOLD = 0.15
JACCARD_K_VALUES = [3, 5]

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]
CRITICAL_PAIRS = {"cat_dog": (3, 5), "cat_deer": (3, 4), "dog_deer": (5, 4)}

# ── Phase-4 hyperparameters (report §3.3) ──────────────────────────────────
PHASE4_LR, PHASE4_MOMENTUM, PHASE4_WEIGHT_DECAY = 0.1, 0.9, 5e-4
PHASE4_N_EPOCHS      = 200
PHASE4_BATCH_SIZE    = 128
PHASE4_LR_MILESTONES = [60, 120, 160]
PHASE4_LR_DECAY      = 0.1
PHASE4_GAMMA_KD      = 3000
PHASE4_GRAD_CLIP, PHASE4_GRAD_CLIP_EPOCHS = 5.0, 10
PHASE4_WEIGHT_FLOOR  = 0.05
PHASE4_SEEDS         = [42, 123, 456]
PHASE4_CKA_INTERVAL  = 20
PHASE4_KD_SOFT_TEMP, PHASE4_BETA_KD_SOFT = 4.0, 1.0
PHASE4_TEACHER_MATCH_IDX = {"layer1": 2, "layer2": 3, "layer3": 5, "layer4": 2}
PHASE4_STUDENT_MATCH_IDX = {"layer1": 1, "layer2": 1, "layer3": 1, "layer4": 1}

TEACHER_HF_REPO = "edadaltocg/resnet50_cifar10"
TEACHER_CKPT    = CKPT_DIR / "pytorch_model.bin"

# ── Execution flags ────────────────────────────────────────────────────────
RUN_METRICS                   = True   # Phase 2: BIgeo/BIacc/BIrep(+gram/class), 16 blocks
RUN_DEEP_DIVE                 = True   # Phase 3: silent-failure deep dive of the primary candidate
RUN_PER_CLASS_CKA_ALL_BLOCKS  = True   # Phase 3.6: 16 blocks × 10 classes per-class CKA
RUN_PRUNING_REAL              = True   # Real progressive pruning, k=1..16 × 3 orders
RUN_PHASE4_TRAINING           = False  # Full SP-KD re-training (~hours per run!) — see §6
PHASE4_CONDITIONS_TO_TRAIN    = ["vanilla", "uniform"]   # subset used when re-training
PHASE4_SEEDS_TO_TRAIN         = [42]
ADAPTIVE_GAMMA                = False  # post-hoc γ-halving fix (repo commit d2f425a);
                                       # False = faithful to the 12 archived runs
RUN_CIFAR10C_EVAL             = False  # Live CIFAR-10-C/P robustness eval (downloads ~2.9 GB from Zenodo)

def banner(msg):
    print("\n" + "═" * 66 + f"\n{msg}\n" + "═" * 66)

print(f"torch {torch.__version__} | device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
assert TARGET_BLOCKS == [f"layer{s}.{i}" for s, n in zip("1234", (3, 4, 6, 3)) for i in range(n)]

### Shared utilities (port of `utils.py`)

`ablated_block` implements virtual lesion by monkey-patching the block's
forward and restoring it in a `finally` clause. For the four transition blocks
(`layer{1,2,3,4}.0`) the **downsample projection branch is preserved** so tensor
shapes stay valid downstream — only the residual transformation is zeroed.

In [ ]:
def set_seed(seed: int = SEED) -> None:
    """Fix every RNG source; cuDNN deterministic mode (validity requirement)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
    except Exception as e:                      # availability / platform guards
        print(f"  [warn] deterministic algorithms not fully enforceable here: {e}")
    print(f"Global seed set to {seed} (cudnn.deterministic=True, benchmark=False).")


def relax_determinism_for_training() -> None:
    """cuDNN benchmark mode for Phase-4 training only (deterministic mode costs 5–10×)."""
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True
    try:
        torch.use_deterministic_algorithms(False)
    except Exception:
        pass


def worker_init_fn(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def get_dataloader_generator(seed: int = SEED) -> torch.Generator:
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def tensor_checksum(t: torch.Tensor) -> str:
    """SHA-256 of the raw bytes — integrity check on reference features."""
    return hashlib.sha256(t.cpu().numpy().tobytes()).hexdigest()[:16]


@contextmanager
def ablated_block(block, is_downsampling: bool = False):
    """
    Temporarily short-circuit a Bottleneck block.

    Standard block  : output = input                       (identity forward)
    Transition block: output = downsample(input)           (projection preserved)
    Restoration is enforced in `finally` — no state leakage on exceptions.
    """
    original_forward = block.forward
    if not is_downsampling:
        def patched_forward(x):
            return x
    else:
        def patched_forward(x):
            identity = block.downsample(x) if block.downsample is not None else x
            return identity
    block.forward = patched_forward
    try:
        yield
    finally:
        block.forward = original_forward


class ActivationCapture:
    """Forward hooks capturing a module's input and output; removed via .remove()."""
    def __init__(self, module):
        self.input, self.output = None, None
        self._h_in  = module.register_forward_pre_hook(self._cap_in)
        self._h_out = module.register_forward_hook(self._cap_out)

    def _cap_in(self, module, args):
        self.input = args[0].detach()

    def _cap_out(self, module, args, output):
        self.output = output.detach()

    def remove(self):
        self._h_in.remove()
        self._h_out.remove()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.remove()


def gap(tensor: torch.Tensor) -> torch.Tensor:
    """Global average pooling over (H, W): (N,C,H,W) → (N,C). 2-D input passes through."""
    return tensor.mean(dim=[2, 3]) if tensor.dim() == 4 else tensor

<a id="s1"></a>
## 1 · Data & teacher model (Phase 1)

**Data.** CIFAR-10 with standard normalisation (no augmentation — inference only).
The **calibration set** is a class-balanced subset of the *training* split
(250/class ⇒ N = 2500): it feeds $F_{intact}$ and every representational metric,
keeping test data out of the measurement path.

**Teacher.** ResNet-50 with the CIFAR stem (3×3 stride-1 conv, no max-pool,
10-class head — 23.52 M params), frozen weights
`edadaltocg/resnet50_cifar10`, expected test accuracy **0.9464** (report §5.1).
Phase 1 ends by extracting the intact pre-classifier features
$F_{intact}\in\mathbb{R}^{2500\times2048}$ (GAP output, hook-based) whose SHA-256
checksum is stored for integrity.

In [ ]:
# ── Data pipeline (port of data.py) ─────────────────────────────────────────
def get_transform():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    ])

def get_train_transform():
    """Standard CIFAR-10 augmentation — Phase-4 student training only."""
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
    ])

def load_cifar10(data_dir=DATA_DIR):
    tf = get_transform()
    train_ds = datasets.CIFAR10(str(data_dir), train=True,  download=True, transform=tf)
    test_ds  = datasets.CIFAR10(str(data_dir), train=False, download=True, transform=tf)
    print(f"CIFAR-10 loaded — train: {len(train_ds)}, test: {len(test_ds)}")
    return train_ds, test_ds

def build_calibration_indices(train_ds, n_per_class=SAMPLES_PER_CLASS, seed=SEED,
                              save_path=RESULTS_DIR / "calib_indices.pt"):
    """Class-balanced calibration subset; indices saved for exact repeatability."""
    rng = np.random.default_rng(seed)
    targets = np.array(train_ds.targets)
    selected = []
    for cls in range(10):
        cls_indices = np.where(targets == cls)[0]
        selected.extend(rng.choice(cls_indices, size=n_per_class, replace=False).tolist())
    indices = torch.tensor(selected, dtype=torch.long)
    torch.save(indices, save_path)
    print(f"Calibration set: {len(indices)} samples ({n_per_class}/class) → {save_path.name}")
    return indices

def _make_loader(dataset, batch_size, shuffle=False, seed=SEED):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn,
                      generator=get_dataloader_generator(seed) if shuffle else None,
                      persistent_workers=NUM_WORKERS > 0)

def get_test_loader(test_ds):
    return _make_loader(test_ds, BATCH_SIZE_INFERENCE)

def get_calibration_loader(train_ds, calib_indices):
    return _make_loader(Subset(train_ds, calib_indices.tolist()), BATCH_SIZE_CALIB)

def get_class_conditional_loaders(train_ds, calib_indices):
    """One loader per CIFAR-10 class over its calibration samples (per-class CKA)."""
    targets = np.array(train_ds.targets)
    calib_targets = targets[calib_indices.numpy()]
    loaders = {}
    for cls in range(10):
        cls_global = calib_indices[calib_targets == cls].tolist()
        loaders[cls] = _make_loader(Subset(train_ds, cls_global), BATCH_SIZE_CALIB)
    return loaders

In [ ]:
# ── Teacher model (port of model.py + HF-hub acquisition) ──────────────────
class ResNet50_CIFAR(ResNet):
    """ResNet-50 with CIFAR stem: conv1 3×3 stride-1, maxpool → Identity, 10 classes."""
    def __init__(self, num_classes: int = 10):
        super().__init__(block=Bottleneck, layers=[3, 4, 6, 3], num_classes=num_classes)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.maxpool = nn.Identity()


class ResNet18_CIFAR(ResNet):
    """ResNet-18 student with the same stem modification (11.17 M params)."""
    def __init__(self, num_classes: int = 10):
        super().__init__(block=BasicBlock, layers=[2, 2, 2, 2], num_classes=num_classes)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.maxpool = nn.Identity()

def build_student(num_classes: int = 10):
    return ResNet18_CIFAR(num_classes=num_classes)


def _strip_prefix(state_dict, prefixes=("module.", "net.")):
    cleaned = {}
    for k, v in state_dict.items():
        new_k = k
        for p in prefixes:
            if new_k.startswith(p):
                new_k = new_k[len(p):]
                break
        cleaned[new_k] = v
    return cleaned


def download_teacher_weights(dst: Path = TEACHER_CKPT) -> Path:
    """Fetch the frozen teacher checkpoint (same file used for all report numbers)."""
    if dst.exists() and dst.stat().st_size > 10_000_000:
        return dst
    # 1) huggingface_hub
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo_id=TEACHER_HF_REPO, filename="pytorch_model.bin",
                            local_dir=str(dst.parent))
        return Path(p)
    except Exception as e:
        print(f"[hf_hub download unavailable: {e}] — falling back to direct URL")
    # 2) direct URL
    url = f"https://huggingface.co/{TEACHER_HF_REPO}/resolve/main/pytorch_model.bin"
    urllib.request.urlretrieve(url, dst)
    return dst


def load_teacher(ckpt_path: Path = TEACHER_CKPT, device: torch.device = DEVICE):
    model = ResNet50_CIFAR(num_classes=10)
    if not Path(ckpt_path).exists():
        raise FileNotFoundError(
            f"Checkpoint not found: {ckpt_path}\n"
            "Download failed? Upload pytorch_model.bin manually to /content/checkpoints/.")
    raw = torch.load(ckpt_path, map_location="cpu")
    sd = raw.get("state_dict", raw.get("model", raw)) if isinstance(raw, dict) else raw
    sd = _strip_prefix(sd)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    # fc.* keys exist in both architectures → nothing meaningful may be missing
    assert not missing, f"Missing keys: {missing[:5]} …"
    model = model.to(device).eval()
    print(f"Teacher loaded from {Path(ckpt_path).name} → {device}")
    return model


def build_block_registry(model):
    registry = {}
    for stage_name in STAGES:
        for idx, blk in enumerate(getattr(model, stage_name)):
            registry[f"{stage_name}.{idx}"] = blk
    return registry


@torch.no_grad()
def evaluate_accuracy(model, loader, device=DEVICE) -> float:
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    return correct / total


@torch.no_grad()
def evaluate_accuracy_with_logits(model, loader, device=DEVICE):
    """Accuracy + per-sample softmax probs + labels (confidence analyses)."""
    all_probs, all_labels, correct, total = [], [], 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        probs = torch.softmax(logits, dim=1)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())
    return correct / total, torch.cat(all_probs), torch.cat(all_labels)


@torch.no_grad()
def extract_features_with_labels(model, loader, device=DEVICE):
    """Single-pass GAP features (N×D, CPU) + labels — the workhorse extractor."""
    feats, labels_all = [], []
    with ActivationCapture(model.avgpool) as cap:
        for images, labels in loader:
            images = images.to(device)
            model(images)
            feats.append(gap(cap.output).cpu())
            labels_all.append(labels.cpu())
    return torch.cat(feats, dim=0), torch.cat(labels_all, dim=0)

In [ ]:
# ── Phase 1 execution ───────────────────────────────────────────────────────
banner("PHASE 1 — Setup & Baseline")
t0 = time.time()
set_seed(SEED)

train_ds, test_ds = load_cifar10()
calib_indices = build_calibration_indices(train_ds)
test_loader   = get_test_loader(test_ds)
calib_loader  = get_calibration_loader(train_ds, calib_indices)

teacher = load_teacher(download_teacher_weights(), DEVICE)
registry = build_block_registry(teacher)

baseline_acc = evaluate_accuracy(teacher, test_loader, DEVICE)
print(f"\nSanity check — intact test accuracy: {baseline_acc:.4f} "
      f"(paper: {BASELINE_ACCURACY_EXPECTED:.4f})")
if baseline_acc < ACCURACY_FAIL_THRESHOLD:
    raise RuntimeError("Baseline accuracy far below expectation — check checkpoint/normalisation.")

# Intact reference representations F_intact (N×2048) + checksum
F_intact, labels_intact = extract_features_with_labels(teacher, calib_loader, DEVICE)
assert F_intact.shape == (N_CALIBRATION, 2048), F_intact.shape
ref_meta = {
    "shape": list(F_intact.shape),
    "checksum": tensor_checksum(F_intact),
    "baseline_accuracy": baseline_acc,
    "seed": SEED,
}
torch.save(F_intact, RESULTS_DIR / "reference_representations.pt")
with open(RESULTS_DIR / "reference_meta.json", "w") as f:
    json.dump(ref_meta, f, indent=2)
print(f"F_intact {tuple(F_intact.shape)} checksum={ref_meta['checksum']} "
      f"(local-run checksum was 5689653e2f797f33 — exact bytes depend on torch/GPU)")
print(f"Phase 1 done in {time.time()-t0:,.0f}s")

<a id="s2"></a>
## 2 · Block Influence metrics (Phase 2)

Each of the 16 Bottleneck blocks receives three primary scores (report §2.4):

$$\mathrm{BI}_{geo}(\ell)=1-\tfrac{1}{N}\sum_i \cos\!\big(\mathrm{GAP}(x^{(i)}_{\ell,in}),\,\mathrm{GAP}(x^{(i)}_{\ell,out})\big)$$
(passive, one forward pass — the ShortGPT-style redundancy proxy),

$$\mathrm{BI}_{acc}(\ell)=\mathrm{Acc}(f)-\mathrm{Acc}(f_{-\ell})\qquad\text{(full 10k test set)},$$
$$\mathrm{BI}_{rep}(\ell)=1-\mathrm{CKA}\big(F_{intact},\,F_{-\ell}\big)\qquad\text{(calibration set)},$$

plus two triangulation variants computed from the same ablated features:
$\mathrm{BI}_{rep\text{-}gram}=1-\mathrm{CKA}_\text{HSIC}(K_{intact},K_{-\ell})$
(double-centred $N\times N$ Gram route; mathematically equivalent for the linear
kernel — retained as implementation cross-check) and
$\mathrm{BI}_{rep\text{-}class}=1-\cos\!\big(\mathrm{vec}(S_{intact}),\mathrm{vec}(S_{-\ell})\big)$
on the 10×10 class-centroid cosine matrices $S$.

**CKA caveat handled by design (Ding et al., 2021):** CKA magnitude is never read
as importance alone — $\Delta=\mathrm{BI}_{rep}-\mathrm{BI}_{acc}$ is a
*disagreement signal*, later confirmed output-level (§3).

In [ ]:
# ── CKA family (port of phase2_metrics.py / bi_rep_extended.py) ────────────
def linear_cka(X: torch.Tensor, Y: torch.Tensor, eps: float = CKA_EPSILON) -> float:
    """
    Centred linear CKA (Kornblith et al., 2019):
        CKA(X,Y) = ||Y^T X||_F^2 / (||X^T X||_F ||Y^T Y||_F)
    Column mean-centring is non-negotiable: without it CKA tracks mean
    activation magnitudes rather than geometry.
    """
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)
    numerator   = torch.norm(X.T @ Y, p="fro") ** 2
    denominator = torch.norm(X.T @ X, p="fro") * torch.norm(Y.T @ Y, p="fro") + eps
    return (numerator / denominator).item()


def gram_matrix(F_mat: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Row-normalised N×N Gram matrix G = F̃ F̃^T (used by SP-KD loss too)."""
    N = F_mat.size(0)
    flat = F_mat.view(N, -1)
    norm = flat.norm(dim=1, keepdim=True).clamp(min=eps)
    unit = flat / norm
    return unit @ unit.t()


def gram_cka(F_a: torch.Tensor, F_b: torch.Tensor, eps: float = CKA_EPSILON) -> float:
    """HSIC-formulation CKA on double-centred N×N linear kernels."""
    K_a = F_a - F_a.mean(dim=0, keepdim=True)
    K_b = F_b - F_b.mean(dim=0, keepdim=True)
    K_a, K_b = K_a @ K_a.t(), K_b @ K_b.t()
    N = K_a.shape[0]
    H = torch.eye(N, device=K_a.device, dtype=K_a.dtype) \
        - torch.ones(N, N, device=K_a.device, dtype=K_a.dtype) / N
    K_ac, K_bc = H @ K_a @ H, H @ K_b @ H
    hsic_ab = (K_ac * K_bc).sum()
    denom = ((K_ac * K_ac).sum() * (K_bc * K_bc).sum()).clamp(min=0).sqrt() + eps
    return (hsic_ab / denom).clamp(0.0, 1.0).item()


def class_cosine_matrix(F_mat: torch.Tensor, labels: torch.Tensor, n_classes: int = 10):
    """10×10 cosine-similarity matrix between class centroids."""
    means = torch.zeros(n_classes, F_mat.shape[1], device=F_mat.device)
    for c in range(n_classes):
        mask = labels == c
        if mask.any():
            means[c] = F_mat[mask].mean(dim=0)
    normed = means / means.norm(dim=1, keepdim=True).clamp(min=1e-8)
    return normed @ normed.t()


def class_matrix_similarity(S_intact, S_ablated, eps: float = CKA_EPSILON) -> float:
    """cos(vec(S_intact), vec(S_ablated)) — structural agreement of class geometry."""
    va, vb = S_intact.flatten().float(), S_ablated.flatten().float()
    return ((va * vb).sum() / (va.norm() * vb.norm() + eps)).item()


@torch.no_grad()
def compute_bi_geo(model, registry_, calib_loader_, device=DEVICE):
    """Step 2.1 — passive geometric transformation per block (hook-based)."""
    bi_geo = {}
    for block_name in tqdm(TARGET_BLOCKS, desc="BIgeo"):
        block = registry_[block_name]
        is_ds = block_name in DOWNSAMPLING_BLOCKS
        cos_sims = []
        with ActivationCapture(block) as cap:
            for images, _ in calib_loader_:
                images = images.to(device)
                model(images)
                x_in, x_out = cap.input, cap.output
                if is_ds and block.downsample is not None:
                    x_in = block.downsample(x_in)          # shared dimensionality
                cos_sims.append(F.cosine_similarity(gap(x_in), gap(x_out), dim=1).cpu())
        bi_geo[block_name] = round(1.0 - torch.cat(cos_sims).mean().item(), 6)
    return bi_geo

In [ ]:
# ── Phase 2 execution — one ablation loop for ALL representation metrics ───
banner("PHASE 2 — Block Influence Metrics (16 blocks)")
t0 = time.time()
set_seed(SEED)

bi_geo = compute_bi_geo(teacher, registry, calib_loader, DEVICE)

# Intact class-cosine structure (computed once, reused by BIrep_class)
S_intact = class_cosine_matrix(F_intact.to(DEVICE), labels_intact.to(DEVICE))

bi_acc, bi_rep, bi_rep_gram, bi_rep_class = {}, {}, {}, {}
pbar = tqdm(TARGET_BLOCKS, desc="BIacc+BIrep")
for block_name in pbar:
    block = registry[block_name]
    is_ds = block_name in DOWNSAMPLING_BLOCKS

    with torch.no_grad(), ablated_block(block, is_downsampling=is_ds):
        # Pass A — full test set → BIacc
        acc_abl = evaluate_accuracy(teacher, test_loader, DEVICE)
        # Pass B — calibration set → F_abl feeds BIrep, BIrep_gram, BIrep_class
        F_abl, _ = extract_features_with_labels(teacher, calib_loader, DEVICE)

    bi_acc[block_name] = round(baseline_acc - acc_abl, 6)
    cka = linear_cka(F_intact, F_abl)
    bi_rep[block_name] = round(1.0 - cka, 6)
    bi_rep_gram[block_name] = round(
        1.0 - gram_cka(F_intact.to(DEVICE), F_abl.to(DEVICE)), 6)
    S_abl = class_cosine_matrix(F_abl.to(DEVICE), labels_intact.to(DEVICE))
    bi_rep_class[block_name] = round(1.0 - class_matrix_similarity(S_intact, S_abl), 6)
    del F_abl
    pbar.set_postfix({"BIacc": bi_acc[block_name], "BIrep": bi_rep[block_name]})

delta_map = {b: round(bi_rep[b] - bi_acc[b], 6) for b in TARGET_BLOCKS}

# Sanity checks (Step 2.4): ranges, anomalies, monotonicity heuristic
phase2_warnings = []
for b in TARGET_BLOCKS:
    if not (0.0 <= bi_rep[b] <= 1.0):
        phase2_warnings.append(f"BIrep[{b}] out of range")
    if bi_acc[b] < -0.01:
        phase2_warnings.append(f"BIacc[{b}] negative (ablation improved accuracy)")
max_dev = max(abs(bi_rep[b] - bi_rep_gram[b]) for b in TARGET_BLOCKS)
print(f"Cross-check |BIrep − BIrep_gram| max = {max_dev:.2e} "
      f"(paper reports < 3e-5 across all 16 blocks)")
print(f"Warnings: {phase2_warnings or 'none'}")

phase2_results = {
    "metadata": {"model": "ResNet50-CIFAR", "seed": SEED, "device": str(DEVICE),
                 "baseline_accuracy": baseline_acc,
                 "n_calibration": N_CALIBRATION},
    "bi_geo": bi_geo, "bi_acc": bi_acc, "bi_rep": bi_rep,
    "bi_rep_gram": bi_rep_gram, "bi_rep_class": bi_rep_class, "delta": delta_map,
    "s_intact_10x10": S_intact.cpu().tolist(),
    "warnings": phase2_warnings,
}
with open(RESULTS_DIR / "phase2_results.json", "w") as f:
    json.dump(phase2_results, f, indent=2)

df_p2 = pd.DataFrame({b: {"BIgeo": bi_geo[b], "BIacc": bi_acc[b], "BIrep": bi_rep[b],
                          "BIrep_gram": bi_rep_gram[b], "BIrep_class": bi_rep_class[b],
                          "Δ=BIrep−BIacc": delta_map[b]} for b in TARGET_BLOCKS}).T
print(f"\nPhase 2 done in {time.time()-t0:,.0f}s — Table 1 of the report:")
display(df_p2.round(4))

<a id="s3"></a>
## 3 · Comparative analysis & the layer4.0 silent failure (Phase 3)

Steps (report E1/E2, §5.1):

1. **Agreement:** Kendall's $\tau$ between metric pairs (+ p-values) and
   top-k Jaccard overlap (k ∈ {3,5}, tie-aware).
2. **Dissociation:** tercile thresholds
   ($\theta_{acc}$ = lower tercile of BIacc, $\theta_{rep}$ = upper tercile of
   BIrep) flag *silent-failure candidates*: low behavioural damage + high
   representational damage, ranked by $\Delta(\ell)=\mathrm{BI}_{rep}-\mathrm{BI}_{acc}$.
3. **Behavioural confirmation** (no CKA in the loop): softmax entropy and top-1
   confidence on still-correct samples, paired one-sided Wilcoxon tests;
   per-class CKA; class-pair centroid-merger analysis.

Note on candidate selection (faithful to the executed protocol): under the strict
joint tercile criteria no block qualifies (layer4.0's BIacc = 0.037 sits above
θ_acc ≈ 0.011), so the pipeline's **null-result fallback** selects the highest-Δ
block — which is `layer4.0`, exactly as in the original run
(`phase3_results.json`: primary `['layer4.0']`).

In [ ]:
# ── Steps 3.1 / 3.2 — rank correlation & top-k Jaccard (port) ──────────────
def compute_rank_correlations(metrics: dict) -> dict:
    pairs = [("bi_geo", "bi_acc"), ("bi_geo", "bi_rep"), ("bi_acc", "bi_rep")]
    results = {}
    for m1, m2 in pairs:
        v1 = [metrics[m1][b] for b in TARGET_BLOCKS]
        v2 = [metrics[m2][b] for b in TARGET_BLOCKS]
        tau, p_val = stats.kendalltau(v1, v2)
        results[f"{m1}_vs_{m2}"] = {"tau": round(float(tau), 4),
                                    "p_value": round(float(p_val), 6),
                                    "significant": bool(p_val < ALPHA)}
    return results


def _top_k_with_ties(scores: dict, k: int) -> set:
    sorted_blocks = sorted(scores, key=lambda b: scores[b], reverse=True)
    threshold = scores[sorted_blocks[k - 1]] if len(sorted_blocks) >= k else -float("inf")
    return {b for b in scores if scores[b] >= threshold and
            sorted_blocks.index(b) < k or scores[b] == threshold}


def _jaccard(Aset, Bset) -> float:
    return 1.0 if not Aset and not Bset else len(Aset & Bset) / len(Aset | Bset)


def compute_jaccard(metrics: dict, k_values=JACCARD_K_VALUES) -> dict:
    results = {}
    for k in k_values:
        top_sets = {name: _top_k_with_ties(scores, k) for name, scores in metrics.items()}
        entry = {}
        for m1, m2 in [("bi_geo", "bi_acc"), ("bi_geo", "bi_rep"), ("bi_acc", "bi_rep")]:
            s1, s2 = top_sets[m1], top_sets[m2]
            entry[f"{m1}_vs_{m2}"] = {"jaccard": round(_jaccard(s1, s2), 4),
                                      "intersection": sorted(s1 & s2)}
        results[str(k)] = entry
    return results


metrics3 = {"bi_geo": bi_geo, "bi_acc": bi_acc, "bi_rep": bi_rep}
corr_results = compute_rank_correlations(metrics3)
jaccard_results = compute_jaccard(metrics3)

tau_table = pd.DataFrame(corr_results).T.rename(index={
    "bi_geo_vs_bi_acc": "τ(BIgeo, BIacc)", "bi_geo_vs_bi_rep": "τ(BIgeo, BIrep)",
    "bi_acc_vs_bi_rep": "τ(BIacc, BIrep)"})
print("Kendall's τ (paper §5.1: 0.85 / 0.633 / 0.683, all significant):")
display(tau_table)
for k, entry in jaccard_results.items():
    for pair, v in entry.items():
        print(f"J(k={k}) {pair}: {v['jaccard']:.2f}  ∩={v['intersection']}")
with open(RESULTS_DIR / "phase3_correlations.json", "w") as f:
    json.dump(corr_results, f, indent=2)
with open(RESULTS_DIR / "phase3_jaccard.json", "w") as f:
    json.dump(jaccard_results, f, indent=2)

In [ ]:
# ── Step 3.3.1 — silent-failure candidate identification (port) ────────────
acc_vals = np.array([bi_acc[b] for b in TARGET_BLOCKS])
rep_vals = np.array([bi_rep[b] for b in TARGET_BLOCKS])
theta_acc = np.percentile(acc_vals, 33.3)     # lower tercile
theta_rep = np.percentile(rep_vals, 66.7)     # upper tercile
print(f"θ_acc = {theta_acc:.4f}   θ_rep = {theta_rep:.4f}")

sorted_by_delta = sorted(delta_map, key=lambda b: delta_map[b], reverse=True)
candidates = [b for b in sorted_by_delta if bi_acc[b] < theta_acc and bi_rep[b] > theta_rep]
if not candidates:                                   # null-result protocol
    print("No block satisfies both criteria — falling back to highest-Δ block.")
    primary, secondary = [sorted_by_delta[0]], []
else:
    primary = [candidates[0]]
    secondary = [b for b in candidates[1:]
                 if delta_map[b] > SECONDARY_DELTA_THRESHOLD][:2]

print(f"Primary candidate  : {primary}   (paper: ['layer4.0'], Δ = 0.098)")
print(f"Secondary candidates: {secondary}")
for b in primary + secondary:
    print(f"  {b:12s} BIacc={bi_acc[b]:.4f}  BIrep={bi_rep[b]:.4f}  Δ={delta_map[b]:.4f}")

In [ ]:
# ── Steps 3.3.2 / 3.3.3 — deep dive of the primary candidate ───────────────
def shannon_entropy(probs):
    p = probs.clamp(min=1e-10)
    return -(p * p.log()).sum(dim=1)


@torch.no_grad()
def analyse_decision_confidence(block_name):
    """Entropy/confidence on still-correct samples C_l, paired Wilcoxon (one-sided)."""
    block = registry[block_name]
    is_ds = block_name in DOWNSAMPLING_BLOCKS
    acc_i, probs_i, labels = evaluate_accuracy_with_logits(teacher, test_loader, DEVICE)
    with ablated_block(block, is_downsampling=is_ds):
        acc_a, probs_a, _ = evaluate_accuracy_with_logits(teacher, test_loader, DEVICE)
    preds_i, preds_a = probs_i.argmax(1), probs_a.argmax(1)
    C_l = (preds_a == labels) & (preds_i == labels)

    H_i, H_a = shannon_entropy(probs_i[C_l]), shannon_entropy(probs_a[C_l])
    stat_h, p_h = stats.wilcoxon(H_a.numpy(), H_i.numpy(), alternative="greater")
    conf_i, conf_a = probs_i[C_l].max(1).values, probs_a[C_l].max(1).values
    _, p_c = stats.wilcoxon(conf_i.numpy(), conf_a.numpy(), alternative="greater")
    out = {
        "block": block_name, "acc_intact": round(acc_i, 6), "acc_ablated": round(acc_a, 6),
        "n_C_l": int(C_l.sum()),
        "mean_H_intact": round(float(H_i.mean()), 6), "mean_H_ablated": round(float(H_a.mean()), 6),
        "mean_delta_H": round(float((H_a - H_i).mean()), 6),
        "median_delta_H": round(float((H_a - H_i).median()), 6),
        "wilcoxon_p_entropy": round(float(p_h), 6),
        "mean_conf_intact": round(float(conf_i.mean()), 6),
        "mean_conf_ablated": round(float(conf_a.mean()), 6),
        "wilcoxon_p_confidence": round(float(p_c), 6),
        "_H_intact": H_i, "_H_ablated": H_a,
    }
    sf[out["block"]] = out
    return out


sf = {}          # block_name → confidence analysis dict (kept in memory for Fig 7)
deep_dive = {}
for block_name in primary + secondary:
    conf_res = analyse_decision_confidence(block_name)
    e = conf_res
    print(f"\n[{block_name}] acc {e['acc_intact']:.4f} → {e['acc_ablated']:.4f} | "
          f"|C_l|={e['n_C_l']}")
    print(f"  entropy  {e['mean_H_intact']:.4f} → {e['mean_H_ablated']:.4f} nats "
          f"(Δ={e['mean_delta_H']:+.4f}, median {e['median_delta_H']:+.4f}, "
          f"Wilcoxon p={e['wilcoxon_p_entropy']:.6f})   ← paper: 0.019→0.354, p<1e-6")
    print(f"  confidence {e['mean_conf_intact']:.4f} → {e['mean_conf_ablated']:.4f} "
          f"(Wilcoxon p={e['wilcoxon_p_confidence']:.6f})")


@torch.no_grad()
def per_class_cka_for_candidate(block_name):
    """Linear CKA per class between intact and ablated representations."""
    block = registry[block_name]
    is_ds = block_name in DOWNSAMPLING_BLOCKS
    class_loaders = get_class_conditional_loaders(train_ds, calib_indices)
    scores = {}
    for cls_idx, cls_loader in class_loaders.items():
        F_cls_intact, _ = extract_features_with_labels(teacher, cls_loader, DEVICE)
        with ablated_block(block, is_downsampling=is_ds):
            F_cls_abl, _ = extract_features_with_labels(teacher, cls_loader, DEVICE)
        scores[str(cls_idx)] = round(linear_cka(F_cls_intact, F_cls_abl), 6)
    return scores


@torch.no_grad()
def class_pair_changes_for_candidate(block_name, top_k=5):
    """Top-k class pairs whose centroid cosine similarity moved most after ablation."""
    block = registry[block_name]
    is_ds = block_name in DOWNSAMPLING_BLOCKS
    with ablated_block(block, is_downsampling=is_ds):
        F_abl, lbls = extract_features_with_labels(teacher, calib_loader, DEVICE)
    S_abl = class_cosine_matrix(F_abl.to(DEVICE), lbls.to(DEVICE))
    pairs = []
    for i in range(10):
        for j in range(i + 1, 10):
            before, after = float(S_intact[i, j]), float(S_abl[i, j])
            pairs.append((abs(after - before), after - before, before, after,
                          CIFAR10_CLASSES[i], CIFAR10_CLASSES[j]))
    pairs.sort(reverse=True)
    return S_abl, pairs[:top_k]


if RUN_DEEP_DIVE:
    cand = primary[0]
    pc_cka = per_class_cka_for_candidate(cand)
    var = float(np.var(list(pc_cka.values())))
    print(f"\nPer-class CKA ({cand}): variance={var:.4f} "
          f"(paper: 0.472 bird … 0.582 ship, var 0.0017)")
    display(pd.Series(pc_cka).rename(index=lambda c: CIFAR10_CLASSES[int(c)]).round(4))

    S_abl_cand, top_pairs = class_pair_changes_for_candidate(cand)
    print(f"\nTop-5 class-pair similarity changes ({cand} ablated) — "
          f"paper: cat↔dog +0.250, ship↔truck +0.217, airplane↔bird +0.214:")
    for abs_d, d, bef, aft, ci, cj in top_pairs:
        arrow = "▲" if d > 0 else "▼"
        print(f"  {ci:>10s} ↔ {cj:<10s} {arrow} {abs_d:.3f}  ({bef:+.4f} → {aft:+.4f})")

    deep_dive = {"per_class_cka": pc_cka, "class_cka_variance": var,
                 "top_pairs": [[ci, cj, round(bef, 6), round(aft, 6)]
                               for _, _, bef, aft, ci, cj in top_pairs]}
    with open(RESULTS_DIR / f"phase3_silent_failure_{cand.replace('.', '_')}.json", "w") as f:
        json.dump(deep_dive, f, indent=2, default=str)

In [ ]:
# ── Figures — metric tables & agreement (Figs 1–4 of the paper) ────────────
METRIC_LABELS = {"bi_geo": "BIgeo", "bi_acc": "BIacc", "bi_rep": "BIrep"}
METRIC_COLORS = {"bi_geo": "#4C72B0", "bi_acc": "#DD8452", "bi_rep": "#55A868"}

def save_fig(fig, name):
    path = FIGURES_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"saved → {path.name}")
    plt.close(fig)


def fig2_grouped_bar():
    order = sorted(TARGET_BLOCKS, key=lambda b: bi_acc[b])
    x = np.arange(len(order)); w = 0.25
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w, [bi_geo[b] for b in order], w, label="BIgeo", color="#4C72B0", alpha=.85)
    ax.bar(x,     [bi_acc[b] for b in order], w, label="BIacc", color="#DD8452", alpha=.85)
    ax.bar(x + w, [bi_rep[b] for b in order], w, label="BIrep", color="#55A868", alpha=.85)
    ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Block Influence score"); ax.axhline(0, color="black", lw=.6, ls="--")
    ax.set_title("Block Influence scores per block (sorted by BIacc)"); ax.legend()
    fig.tight_layout(); save_fig(fig, "fig2_grouped_bar.png")


def fig1_tau_heatmap():
    names = ["bi_geo", "bi_acc", "bi_rep"]; labels = [METRIC_LABELS[n] for n in names]
    mat, pmat = np.eye(3), np.zeros((3, 3))
    pm = {("bi_geo","bi_acc"): "bi_geo_vs_bi_acc", ("bi_geo","bi_rep"): "bi_geo_vs_bi_rep",
          ("bi_acc","bi_rep"): "bi_acc_vs_bi_rep"}
    for i, m1 in enumerate(names):
        for j, m2 in enumerate(names):
            if i != j:
                key = pm.get((m1, m2)) or pm.get((m2, m1))
                mat[i, j], pmat[i, j] = corr_results[key]["tau"], corr_results[key]["p_value"]
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(mat, cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(labels); ax.set_yticks(range(3)); ax.set_yticklabels(labels)
    fig.colorbar(im, ax=ax, label="Kendall's τ")
    for i in range(3):
        for j in range(3):
            sig = "*" if pmat[i, j] < 0.05 else ""
            ax.text(j, i, f"{mat[i,j]:.2f}{sig}", ha="center", va="center",
                    color="black" if abs(mat[i,j]) < .7 else "white")
    ax.set_title("Kendall's τ rank-correlation matrix")
    fig.tight_layout(); save_fig(fig, "fig1_tau_heatmap.png")


def fig3_jaccard_heatmaps():
    fig, axes = plt.subplots(1, len(jaccard_results), figsize=(5 * len(jaccard_results), 4))
    axes = np.atleast_1d(axes)
    names = ["bi_geo", "bi_acc", "bi_rep"]; labels = [METRIC_LABELS[n] for n in names]
    pm = {("bi_geo","bi_acc"): "bi_geo_vs_bi_acc", ("bi_geo","bi_rep"): "bi_geo_vs_bi_rep",
          ("bi_acc","bi_rep"): "bi_acc_vs_bi_rep"}
    for ax, k in zip(axes, jaccard_results):
        mat = np.eye(3)
        for i, m1 in enumerate(names):
            for j, m2 in enumerate(names):
                if i != j:
                    key = pm.get((m1, m2)) or pm.get((m2, m1))
                    mat[i, j] = jaccard_results[k][key]["jaccard"]
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(3)); ax.set_xticklabels(labels)
        ax.set_yticks(range(3)); ax.set_yticklabels(labels)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{mat[i,j]:.2f}", ha="center", va="center")
        ax.set_title(f"k = {k}"); fig.colorbar(im, ax=ax, label="Jaccard")
    fig.suptitle("Top-k Jaccard similarity"); fig.tight_layout()
    save_fig(fig, "fig3_jaccard_heatmaps.png")


fig2_grouped_bar()
fig1_tau_heatmap()
fig3_jaccard_heatmaps()

In [ ]:
# ── Figure 4 — BIacc vs BIrep scatter with tercile thresholds (paper Fig. P3-4)
candidates_all = primary + secondary
fig, ax = plt.subplots(figsize=(7, 6))
for block in TARGET_BLOCKS:
    cand = block in candidates_all
    ax.scatter(bi_acc[block], bi_rep[block], color="crimson" if cand else "#4C72B0",
               s=120 if cand else 50, zorder=3)
    ax.annotate(block, (bi_acc[block], bi_rep[block]), textcoords="offset points",
                xytext=(6, 4), fontsize=7.5,
                color="crimson" if cand else "#333333")
ax.axhline(theta_rep, color="grey", ls="--", lw=.8, label="θ_rep")
ax.axvline(theta_acc, color="grey", ls=":", lw=.8, label="θ_acc")
ax.set_xlabel("BIacc (accuracy drop)"); ax.set_ylabel("BIrep (representational disruption)")
ax.set_title("BIacc vs BIrep — silent failure candidates in red")
ax.legend(fontsize=9); fig.tight_layout()
save_fig(fig, "fig4_scatter.png")

In [ ]:
# ── Figures 7 / 11 / 10 — behavioural signature & class-pair structure ─────
def fig7_entropy(block_name):
    e = sf[block_name]
    means = [e["mean_H_intact"], e["mean_H_ablated"]]
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    bars = ax.bar(["H intact", "H ablated"], means, color=["#4C72B0", "#DD8452"],
                  alpha=.85, width=.45)
    for bar, val in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+.004, f"{val:.4f}",
                ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Mean Shannon entropy (nats)")
    ax.set_title(f"Entropy on still-correct samples C_l — {block_name}\n"
                 f"ΔH_mean={e['mean_delta_H']:+.4f}  "
                 f"Wilcoxon p={e['wilcoxon_p_entropy']:.4f}", fontsize=11)
    fig.tight_layout(); save_fig(fig, f"fig7_entropy_{block_name.replace('.','_')}.png")


def fig11_class_pair_changes(block_name, top_k=5):
    _, pairs = class_pair_changes_for_candidate(block_name, top_k=top_k)
    pair_labels = [f"{ci} ↔ {cj}" for _, _, _, _, ci, cj in pairs]
    befores = [p[2] for p in pairs]; afters = [p[3] for p in pairs]; deltas = [p[1] for p in pairs]
    y = np.arange(top_k); h = .35
    fig, ax = plt.subplots(figsize=(9.5, 5))
    ax.barh(y + h/2, befores, h, label="Before (intact)", color="#4C72B0", alpha=.85)
    ax.barh(y - h/2, afters,  h, label=f"After ({block_name} ablated)", color="#DD8452", alpha=.85)
    for i, (bef, aft, d) in enumerate(zip(befores, afters, deltas)):
        sign = "▲" if d > 0 else "▼"
        ax.text(max(bef, aft)+.01, i, f"{sign} {abs(d):.3f}", va="center", fontsize=9,
                color="#c0392b" if d > 0 else "#1a6b3a", fontweight="bold")
    ax.set_yticks(y); ax.set_yticklabels(pair_labels)
    ax.set_xlabel("Centroid cosine similarity"); ax.set_xlim(0, 1.12)
    ax.set_title(f"Top-{top_k} class-pair similarity changes — {block_name} ablated\n"
                 "All top pairs become MORE similar: class separation degrades silently",
                 fontsize=11)
    ax.legend(); ax.invert_yaxis(); fig.tight_layout()
    save_fig(fig, f"fig11_class_pair_changes_{block_name.replace('.','_')}.png")


def fig10_class_heatmaps(block_name):
    S_int = np.array(phase2_results["s_intact_10x10"])
    S_abl, _lbl = None, None
    block = registry[block_name]; is_ds = block_name in DOWNSAMPLING_BLOCKS
    with torch.no_grad(), ablated_block(block, is_downsampling=is_ds):
        F_abl, lbls = extract_features_with_labels(teacher, calib_loader, DEVICE)
    S_abl = class_cosine_matrix(F_abl.to(DEVICE), lbls.to(DEVICE)).cpu().numpy()
    panels = [(S_int, "Intact", "Blues", 0., 1.),
              (S_abl, f"{block_name} ablated", "Blues", 0., 1.),
              (S_abl - S_int, "Difference (ablated − intact)", "RdBu_r", -.3, .3)]
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    for ax, (mat, title, cmap, vmin, vmax) in zip(axes, panels):
        im = ax.imshow(mat, cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(10)); ax.set_yticklabels(CIFAR10_CLASSES, fontsize=8)
        ax.set_title(title); fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
    fig.suptitle(f"Class×Class centroid cosine similarity — intact vs {block_name} ablated")
    fig.tight_layout(); save_fig(fig, f"fig10_class_heatmaps_{block_name.replace('.','_')}.png")


def fig6_per_class_cka(block_name):
    scores = [pc_cka[str(c)] for c in range(10)]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(CIFAR10_CLASSES, scores, color="#55A868", alpha=.85)
    ax.axhline(np.mean(scores), color="crimson", ls="--", lw=1.2,
               label=f"Mean={np.mean(scores):.3f}")
    ax.set_ylim(0, 1.05); ax.set_ylabel("Linear CKA (intact vs ablated)")
    ax.set_title(f"Per-class CKA disruption — {block_name}")
    ax.legend(); plt.xticks(rotation=30, ha="right"); fig.tight_layout()
    save_fig(fig, f"fig6_per_class_cka_{block_name.replace('.','_')}.png")


if RUN_DEEP_DIVE:
    cand = primary[0]
    fig7_entropy(cand)
    fig11_class_pair_changes(cand)
    fig10_class_heatmaps(cand)
    fig6_per_class_cka(cand)

In [ ]:
# ── Figures 8 / 9 — triangulation cross-checks ─────────────────────────────
def fig8_birep_vs_gram():
    vals_rep  = [bi_rep[b] for b in TARGET_BLOCKS]
    vals_gram = [bi_rep_gram[b] for b in TARGET_BLOCKS]
    lim = max(max(vals_rep), max(vals_gram)) * 1.05
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, lim], [0, lim], color="#bbb", ls="--", lw=1.2, label="y = x (equivalence)")
    sc = ax.scatter(vals_rep, vals_gram, c=vals_rep, cmap="viridis", s=90,
                    edgecolors="white", linewidths=.5, zorder=3)
    for b in TARGET_BLOCKS:
        if abs(bi_rep[b] - bi_rep_gram[b]) > 1e-3 or bi_rep[b] > .1:
            ax.annotate(b, (bi_rep[b], bi_rep_gram[b]), textcoords="offset points",
                        xytext=(5, 3), fontsize=7)
    fig.colorbar(sc, ax=ax, label="BIrep value")
    ax.set_xlabel("BIrep (linear CKA on N×D)"); ax.set_ylabel("BIrep_gram (HSIC CKA on N×N)")
    ax.set_title("BIrep vs BIrep_gram\nmathematical equivalence for the linear kernel", fontsize=11)
    ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.set_aspect("equal"); ax.legend(fontsize=9)
    fig.tight_layout(); save_fig(fig, "fig8_birep_vs_gram.png")


def fig9_birep_class_bar():
    order = sorted(TARGET_BLOCKS, key=lambda b: bi_acc[b])
    x = np.arange(len(order)); w = .35
    fig, ax1 = plt.subplots(figsize=(14, 5)); ax2 = ax1.twinx()
    ax1.bar(x - w/2, [bi_rep[b] for b in order], w, label="BIrep (instance)",
            color="#55A868", alpha=.85)
    ax2.bar(x + w/2, [bi_rep_class[b] for b in order], w, label="BIrep_class (class)",
            color="#C44E52", alpha=.85)
    ax1.set_xticks(x); ax1.set_xticklabels(order, rotation=45, ha="right", fontsize=9)
    ax1.set_ylabel("BIrep — instance-level", color="#55A868")
    ax2.set_ylabel("BIrep_class — class-level", color="#C44E52")
    ax1.set_title("BIrep vs BIrep_class (sorted by BIacc)\ncentroid-level angular structure "
                  "is ~10× more robust to ablation", fontsize=11)
    l1, la1 = ax1.get_legend_handles_labels(); l2, la2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, la1 + la2, loc="upper left", fontsize=9)
    fig.tight_layout(); save_fig(fig, "fig9_birep_class_bar.png")


fig8_birep_vs_gram()
fig9_birep_class_bar()

In [ ]:
# ── Step 3.5 — simulated progressive pruning (cumulative sums, no GPU) ─────
def analyse_progressive_pruning_simulated():
    order1 = sorted(TARGET_BLOCKS, key=lambda b: bi_acc[b])                     # BIacc ascending
    delta  = {b: bi_rep[b] - bi_acc[b] for b in TARGET_BLOCKS}
    order2 = sorted(TARGET_BLOCKS, key=lambda b: delta[b], reverse=True)        # Δ descending
    order3 = sorted(TARGET_BLOCKS, key=lambda b: bi_rep[b])                     # BIrep ascending
    def cumulative(order, metric):
        running, vals = 0.0, []
        for b in order:
            running += metric[b]
            vals.append(round(running, 6))
        return vals
    res = {"strategy1_order": order1, "strategy2_order": order2, "strategy3_order": order3}
    for i, order in enumerate([order1, order2, order3], 1):
        res[f"strategy{i}_cumulative_biacc"] = cumulative(order, bi_acc)
        res[f"strategy{i}_cumulative_birep"] = cumulative(order, bi_rep)
    return res

simulated_pruning = analyse_progressive_pruning_simulated()
with open(RESULTS_DIR / "phase3_pruning.json", "w") as f:
    json.dump(simulated_pruning, f, indent=2)


def fig12_progressive_pruning():
    strategies = [("strategy1", "Strategy 1: BIacc ascending"),
                  ("strategy2", "Strategy 2: Δ descending (silent-failure first)"),
                  ("strategy3", "Strategy 3: BIrep ascending")]
    n_steps = len(simulated_pruning["strategy1_order"])
    steps = list(range(1, n_steps + 1))
    fig, axes = plt.subplots(1, 3, figsize=(20, 5), constrained_layout=True)
    for ax, (key, title) in zip(axes, strategies):
        order = simulated_pruning[f"{key}_order"]
        ax.plot(steps, simulated_pruning[f"{key}_cumulative_biacc"], marker="o", ms=4,
                color="#e05c5c", label="Cumulative BIacc")
        ax.plot(steps, simulated_pruning[f"{key}_cumulative_birep"], marker="s", ms=4,
                color="#5c8ee0", ls="--", label="Cumulative BIrep")
        ax.set_xticks(steps); ax.set_xticklabels(order, rotation=45, ha="right", fontsize=7)
        ax.set_xlabel("Block removed (left → right)"); ax.set_ylabel("Cumulative loss")
        ax.set_title(title, fontsize=10, fontweight="bold"); ax.legend(fontsize=8)
        ax.grid(True, ls=":", alpha=.5); ax.set_ylim(bottom=0)
    fig.suptitle("Simulated progressive pruning — additive prediction", fontweight="bold")
    save_fig(fig, "fig12_progressive_pruning.png")

fig12_progressive_pruning()

In [ ]:
# ── Step 3.6 — per-class CKA for ALL 16 blocks (Figs 13–14) ────────────────
if RUN_PER_CLASS_CKA_ALL_BLOCKS:
    t0 = time.time()
    class_loaders = get_class_conditional_loaders(train_ds, calib_indices)
    # intact representations per class, extracted once
    F_cls_intact = {}
    with torch.no_grad():
        for cls_idx, loader in class_loaders.items():
            F_cls_intact[cls_idx], _ = extract_features_with_labels(teacher, loader, DEVICE)

    per_class_cka_all = {}
    for block_name in tqdm(TARGET_BLOCKS, desc="per-class CKA"):
        block = registry[block_name]
        is_ds = block_name in DOWNSAMPLING_BLOCKS
        scores = {}
        with torch.no_grad(), ablated_block(block, is_downsampling=is_ds):
            for cls_idx, loader in class_loaders.items():
                F_abl, _ = extract_features_with_labels(teacher, loader, DEVICE)
                scores[cls_idx] = round(linear_cka(F_cls_intact[cls_idx], F_abl), 6)
        per_class_cka_all[block_name] = scores
    with open(RESULTS_DIR / "phase3_per_class_cka.json", "w") as f:
        json.dump(per_class_cka_all, f, indent=2)
    print(f"done in {time.time()-t0:,.0f}s")


def fig13_per_class_cka_heatmap():
    block_names = list(per_class_cka_all.keys())
    data = np.array([[per_class_cka_all[b][c] for c in range(10)] for b in block_names])
    n_blocks = data.shape[0]
    fig, ax = plt.subplots(figsize=(12, max(4, .5 * n_blocks + 2)))
    im = ax.imshow(data, aspect="auto", cmap="viridis", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label="Linear CKA")
    ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45, ha="right", fontsize=9)
    ax.set_yticks(range(n_blocks)); ax.set_yticklabels(block_names, fontsize=8)
    if n_blocks == 16:                       # stage separators (full registry only)
        for sep in (2.5, 6.5, 12.5):
            ax.axhline(sep, color="white", lw=2)
    ax.set_title("Per-class linear CKA (intact vs ablated) — all blocks",
                 fontweight="bold", pad=10)
    fig.tight_layout(); save_fig(fig, "fig13_per_class_cka_heatmap.png")


def fig14_top5_per_class_bar():
    top5_default = ["layer1.0", "layer3.0", "layer2.0", "layer4.0", "layer1.1"]
    top5 = [b for b in top5_default if b in per_class_cka_all][:5]
    fig, axes = plt.subplots(1, len(top5), figsize=(3.6 * len(top5), 4),
                             sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, block_name in zip(axes, top5):
        scores = [per_class_cka_all[block_name][c] for c in range(10)]
        ax.bar(range(10), scores, color="#4C72B0")
        ax.set_xticks(range(10)); ax.set_xticklabels(CIFAR10_CLASSES, rotation=45,
                                                     ha="right", fontsize=8)
        ax.set_title(f"{block_name}\nvar={np.var(scores):.4f}", fontsize=10, fontweight="bold")
        ax.set_ylim(0, 1); ax.grid(True, ls=":", alpha=.4)
    axes[0].set_ylabel("Linear CKA")
    fig.suptitle("Per-class CKA — top-5 blocks by BIrep", fontweight="bold")
    save_fig(fig, "fig14_top5_per_class_bar.png")


if RUN_PER_CLASS_CKA_ALL_BLOCKS:
    fig13_per_class_cka_heatmap()
    fig14_top5_per_class_bar()

<a id="s4"></a>
## 4 · Real progressive pruning (E3, report §5.2)

Marginal (single-block) scores implicitly assume blocks contribute
**independently**. This experiment falsifies that assumption *in the
informative direction*: the first $k$ blocks of each ordering are ablated
**simultaneously** (all $k$ context managers entered through a single
`ExitStack` before any forward pass — atomic ablation), and real damage is
measured against the additive prediction $\sum_{i\le k}\mathrm{BI}^{(i)}$:

* real accuracy, mean top-1 confidence, mean entropy, entropy on still-correct
  samples $H_{C_l}$ (test set);
* real $\mathrm{BIrep}_k = 1-\mathrm{CKA}(F_{intact}, F_{\text{ablated-}k})$ (calibration set).

Paper finding: **superadditivity ≈ 2.2× at k = 9**; $H_{C_l}$ spikes at
k ≈ 12–13 (population-level silent failure).

In [ ]:
_EPS = 1e-10

def _entropy(probs):
    return -(probs * torch.log(probs + _EPS)).sum(dim=1)


@torch.no_grad()
def _test_pass(model, loader, device=DEVICE):
    logits = torch.cat([model(img.to(device)).cpu() for img, _ in loader])
    labels = torch.cat([lbl for _, lbl in loader])
    probs = torch.softmax(logits, dim=1)
    preds = probs.argmax(dim=1)
    correct = preds == labels
    H = _entropy(probs)
    return (correct.float().mean().item(),
            probs.max(dim=1).values.mean().item(),
            H.mean().item(),
            H[correct].mean().item() if correct.any() else 0.0)


@torch.no_grad()
def _calib_pass(model, loader, device=DEVICE):
    feats = []
    with ActivationCapture(model.avgpool) as cap:
        for images, _ in loader:
            model(images.to(device))
            feats.append(gap(cap.output).cpu())
    return torch.cat(feats, dim=0)


def _build_order(metric_key):
    return sorted(TARGET_BLOCKS, key=lambda b: phase2_results[metric_key][b])


def _compute_simulated(order):
    cum_acc = cum_rep = 0.0
    out = {}
    for k, block in enumerate(order, start=1):
        cum_acc += bi_acc[block]; cum_rep += bi_rep[block]
        out[f"k{k}"] = {"cumulative_biacc": round(cum_acc, 6),
                        "cumulative_birep": round(cum_rep, 6)}
    return out


def run_real_progressive_pruning():
    F_ref = F_intact.to(DEVICE).float()
    strategies = {"strategy1": _build_order("bi_acc"),
                  "strategy3": _build_order("bi_rep"),
                  "strategy_geo": _build_order("bi_geo")}
    results = {}
    for strat_key, order in strategies.items():
        print(f"\n─ {strat_key}: {order}")
        real, simulated = {}, _compute_simulated(order)
        for k in tqdm(range(1, len(order) + 1), desc=strat_key):
            blocks_k = order[:k]
            with ExitStack() as stack:
                for bn in blocks_k:
                    stack.enter_context(ablated_block(
                        registry[bn], is_downsampling=(bn in DOWNSAMPLING_BLOCKS)))
                acc, conf_mean, H_mean, H_Cl = _test_pass(teacher, test_loader, DEVICE)
                F_abl = _calib_pass(teacher, calib_loader, DEVICE)
            birep_k = 1.0 - linear_cka(F_ref.cpu(), F_abl)
            real[f"k{k}"] = {"acc": round(acc, 6), "BIrep": round(birep_k, 6),
                             "H_mean": round(H_mean, 6), "conf_mean": round(conf_mean, 6),
                             "H_Cl": round(H_Cl, 6)}
        results[strat_key] = {"order": order, "real": real, "simulated": simulated}
    with open(RESULTS_DIR / "progressive_pruning_real.json", "w") as f:
        json.dump(results, f, indent=2)
    return results

pruning_real = run_real_progressive_pruning() if RUN_PRUNING_REAL else None

In [ ]:
# ── Superadditivity table + Figure (paper Fig. P3-15 / fig_progressive_pruning_real)
STRATEGY_LABELS = {
    "strategy1": "Strategy 1: BIacc ascending",
    "strategy3": "Strategy 3: BIrep ascending",
    "strategy_geo": "Strategy Geo: BIgeo ascending",
}

if pruning_real:
    rows = {}
    for strat_key, strat in pruning_real.items():
        r9, s9 = strat["real"]["k9"], strat["simulated"]["k9"]
        ratio = r9["BIrep"] / max(s9["cumulative_birep"], 1e-9)
        rows[STRATEGY_LABELS[strat_key].split(":")[0]] = {
            "real BIrep(k=9)": r9["BIrep"],
            "additive pred.": s9["cumulative_birep"],
            "ratio": round(ratio, 2),
            "acc(k=10)": strat["real"]["k10"]["acc"],
            "acc(k=11)": strat["real"]["k11"]["acc"],
            "H_Cl peak": max(v["H_Cl"] for v in strat["real"].values()),
        }
    print("Superadditivity at k=9 (paper: 2.20× / 2.35× / 2.32×) and non-monotone accuracy:")
    display(pd.DataFrame(rows).T)


def plot_progressive_pruning_real(results):
    fig, axes = plt.subplots(3, 2, figsize=(18, 15), constrained_layout=True)
    fig.suptitle("Real vs simulated progressive pruning", fontsize=13, fontweight="bold")
    for row, (strat_key, strat) in enumerate(
            ((k, results[k]) for k in ("strategy1", "strategy3", "strategy_geo"))):
        order, real, sim = strat["order"], strat["real"], strat["simulated"]
        ks = list(range(1, len(order) + 1))
        acc_vals   = [real[f"k{k}"]["acc"] for k in ks]
        bireps     = [real[f"k{k}"]["BIrep"] for k in ks]
        conf_vals  = [real[f"k{k}"]["conf_mean"] for k in ks]
        H_means    = [real[f"k{k}"]["H_mean"] for k in ks]
        HCl_vals   = [real[f"k{k}"]["H_Cl"] for k in ks]
        sim_acc    = [sim[f"k{k}"]["cumulative_biacc"] for k in ks]
        sim_rep    = [sim[f"k{k}"]["cumulative_birep"] for k in ks]
        norm_acc = sim_acc[-1] or 1.0
        norm_rep = sim_rep[-1] or 1.0
        sim_acc_norm = [1.0 - v / norm_acc for v in sim_acc]
        sim_rep_norm = [v / norm_rep for v in sim_rep]

        ax0, ax0r = axes[row, 0], axes[row, 0].twinx()
        ln1, = ax0.plot(ks, bireps, color="#5c8ee0", ls="--", lw=1.8, marker="s",
                        ms=4, label="BIrep (real)")
        ln2, = ax0.plot(ks, sim_acc_norm, color="#e05c5c", ls=":", lw=1.5, label="Sim BIacc (norm)")
        ln3, = ax0.plot(ks, sim_rep_norm, color="#5c8ee0", ls=":", lw=1.5, label="Sim BIrep (norm)")
        ln4, = ax0r.plot(ks, acc_vals, color="#e05c5c", lw=2, marker="o", ms=4, label="Acc (real)")
        ax0.set_ylabel("BIrep / normalized simulated"); ax0r.set_ylabel("Accuracy", color="#e05c5c")
        ax0r.tick_params(axis="y", labelcolor="#e05c5c"); ax0r.set_ylim(0, 1.05)
        ax0.set_ylim(bottom=0); ax0.set_xticks(ks)
        ax0.set_xticklabels(order, rotation=45, ha="right", fontsize=7)
        ax0.set_xlabel("Block removed (left → right)")
        ax0.set_title(f"{STRATEGY_LABELS[strat_key]}\nAccuracy & Representation",
                      fontsize=9, fontweight="bold")
        ax0.grid(True, ls=":", alpha=.5)
        lines = [ln1, ln2, ln3, ln4]
        ax0.legend(lines, [l.get_label() for l in lines], fontsize=7, loc="upper left")

        ax1 = axes[row, 1]
        ax1.plot(ks, conf_vals, color="orange", lw=1.8, marker="o", ms=4,
                 label="Confidence (mean top-1)")
        ax1.plot(ks, HCl_vals, color="purple", lw=1.8, marker="^", ms=4,
                 label="H_Cl (entropy, correct samples)")
        ax1.plot(ks, H_means, color="gray", ls="--", lw=1.5, label="H_mean (all samples)")
        ax1.set_xticks(ks); ax1.set_xticklabels(order, rotation=45, ha="right", fontsize=7)
        ax1.set_xlabel("Block removed (left → right)")
        ax1.set_title(f"{STRATEGY_LABELS[strat_key]}\nConfidence & Entropy",
                      fontsize=9, fontweight="bold")
        ax1.legend(fontsize=7); ax1.grid(True, ls=":", alpha=.5); ax1.set_ylim(bottom=0)
    save_fig(fig, "fig_progressive_pruning_real.png")

if pruning_real:
    plot_progressive_pruning_real(pruning_real)

<a id="s5"></a>
## 5 · Phase 4 — BI-weighted SP-KD: anatomy of a null result

**Hypothesis under test (project proposal).** Weighting the Similarity-Preserving
KD loss (Tung & Mori, 2019) by per-stage Block Influence should produce better
ResNet-18 students than uniform weighting. **Result: not confirmed** — all SP-KD
conditions are statistically indistinguishable (report §5.3–5.6).

**Design.** For matched stage pairs $\ell$ (teacher `layer{1.2,2.3,3.5,4.2}` ↔
student `layer{1.1,2.1,3.1,4.1}`, same spatial dims; Gram matrices are $N\times N$
so channel widths may differ):

$$\mathcal{L}_{SP}=\sum_{s} w_s\,\frac{\lVert G_T^{(s)}-G_S^{(s)}\rVert_F^2}{N^2},
\qquad \mathcal{L}=\mathcal{L}_{CE}+\gamma\,\mathcal{L}_{SP},\quad \gamma=3000$$

with $G$ the row-normalised Gram matrix of the flattened activations.
Conditions: **vanilla** (CE only), **uniform** ($w_s=1/4$),
**bi\_acc / bi\_rep** ($w_s \propto$ stage-mean BIacc / BIrep, floor 0.05,
normalised); 3 seeds each.

This section:
1. re-implements the full loss/training machinery;
2. **diagnoses live** the two mechanical causes of the null result
   (weight-vector collapse after stage averaging; $\gamma$ making any normalised
   weighting gradient-inert);
3. regenerates every Phase-4 figure/table of the paper from an embedded snapshot
   of the original run artifacts (`ARCHIVED_PHASE4`);
4. optionally retrains from scratch (`RUN_PHASE4_TRAINING=True`, hours per run)
   or recomputes post-hoc metrics from provided student checkpoints.

In [ ]:
# ── SP-KD machinery (port of phase4_distillation.py) ───────────────────────
def _stage_mean(metric_dict, stage):
    vals = [metric_dict[b] for b in STAGE_BLOCKS[stage] if b in metric_dict]
    return sum(vals) / len(vals) if vals else 0.0


def compute_stage_weights(metric_dict, floor=PHASE4_WEIGHT_FLOOR):
    """Stage-mean → floor → normalise. THE aggregation whose signal-collapse is diagnosed below."""
    raw = {s: max(_stage_mean(metric_dict, s), floor) for s in STAGES}
    total = sum(raw.values())
    return {s: v / total for s, v in raw.items()}


def build_condition_weights(phase2_results_):
    bi_acc_ = phase2_results_["bi_acc"]
    bi_rep_ = phase2_results_["bi_rep"]
    delta = {b: max(bi_rep_.get(b, 0) - bi_acc_.get(b, 0), 0.0) for b in bi_rep_}
    return {
        "vanilla":    {s: 0.0 for s in STAGES},
        "uniform":    {s: 1.0 / len(STAGES) for s in STAGES},
        "bi_acc":     compute_stage_weights(bi_acc_),
        "bi_rep":     compute_stage_weights(bi_rep_),
        "conf_gated": compute_stage_weights(delta),   # Δ-weighted ablations (§5.4)
    }


def sp_kd_loss(teacher_feats, student_feats, weights):
    device = next(iter(student_feats.values())).device
    loss = torch.zeros(1, device=device)
    N = next(iter(student_feats.values())).size(0)
    for stage in STAGES:
        w = weights.get(stage, 0.0)
        if w == 0.0:
            continue
        G_T = gram_matrix(teacher_feats[stage])          # detached constant
        G_S = gram_matrix(student_feats[stage])          # gradients flow here
        diff = G_T - G_S
        loss = loss + w * (diff * diff).sum() / (N * N)
    return loss


def kd_soft_loss(student_logits, teacher_logits, T=PHASE4_KD_SOFT_TEMP):
    """Hinton KD: KL(s/T ‖ t/T)·T²."""
    p_t = F.softmax(teacher_logits / T, dim=1)
    log_p_s = F.log_softmax(student_logits / T, dim=1)
    return (T * T) * F.kl_div(log_p_s, p_t, reduction="batchmean")


class SPHookManager:
    """Hooks on the last block of each matched teacher/student stage."""
    def __init__(self, teacher_mod, student_mod):
        self._t, self._s, self._handles = {}, {}, []
        for stage in STAGES:
            t_mod = getattr(teacher_mod, stage)[PHASE4_TEACHER_MATCH_IDX[stage]]
            s_mod = getattr(student_mod, stage)[PHASE4_STUDENT_MATCH_IDX[stage]]
            self._handles.append(t_mod.register_forward_hook(self._mk_t(stage)))
            self._handles.append(s_mod.register_forward_hook(self._mk_s(stage)))

    def _mk_t(self, name):
        def hook(m, inp, out):
            self._t[name] = out.detach()
        return hook

    def _mk_s(self, name):
        def hook(m, inp, out):
            self._s[name] = out                          # keep autograd graph
        return hook

    @property
    def teacher_feats(self):
        return self._t

    @property
    def student_feats(self):
        return self._s

    def remove(self):
        for h in self._handles:
            h.remove()
        self._handles.clear()


@torch.no_grad()
def extract_avgpool_repr(model, loader, device=DEVICE):
    feats = []
    with ActivationCapture(model.avgpool) as cap:
        for images, _ in loader:
            model(images.to(device))
            feats.append(gap(cap.output).cpu())
    return torch.cat(feats, dim=0)


def compute_transfer_cka(student, teacher_mod, calib_loader_, device=DEVICE):
    student.eval()
    F_s = extract_avgpool_repr(student, calib_loader_, device)
    F_t = extract_avgpool_repr(teacher_mod, calib_loader_, device)
    return linear_cka(F_s, F_t)


@torch.no_grad()
def evaluate_student(model, loader, device=DEVICE):
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += labels.size(0)
    return correct / total

### 5.2 · Archived Phase-4 artifacts

The 12 primary training runs (4 conditions × 3 seeds × 200 epochs ≈ 72 GPU-hours)
cannot be repeated inside one Colab session. The cell below embeds a compact,
verbatim snapshot of the original result files
(`phase4_results/phase4_*.json`, extracted by
`submission/build/extract_archive.py`) so that **every Phase-4 figure and table
of the report can be regenerated anywhere**. Provenance:

* hardware: NVIDIA RTX 4070 Laptop (8 GB) + Kaggle P100 (torch 2.10, Python 3.13);
* students trained from scratch, identical hyperparameters as §5 above;
* `final_cka` = teacher–student linear CKA on the calibration set at epoch 200;
  post-hoc geometry/confidence/robustness statistics from the analysis JSONs.

If you have the original checkpoints, §5.6 recomputes these statistics live.

In [ ]:
# Embedded snapshot of the original Phase-4 run artifacts
# (phase4_results/*.json — see §5.2 markdown for provenance).
ARCHIVED_PHASE4_JSON = (
    '{\n "final_runs": {\n  "bi_acc": {\n   "42": {\n    "final_accuracy": 0.9532,\n    "final_cka'
    '": 0.978885\n   },\n   "123": {\n    "final_accuracy": 0.9528,\n    "final_cka": 0.979406\n  '
    ' },\n   "456": {\n    "final_accuracy": 0.9504,\n    "final_cka": 0.978989\n   }\n  },\n  "bi_'
    'rep": {\n   "42": {\n    "final_accuracy": 0.9527,\n    "final_cka": 0.977514\n   },\n   "123'
    '": {\n    "final_accuracy": 0.9541,\n    "final_cka": 0.978412\n   },\n   "456": {\n    "fina'
    'l_accuracy": 0.9494,\n    "final_cka": 0.978835\n   }\n  },\n  "uniform": {\n   "42": {\n    "'
    'final_accuracy": 0.9519,\n    "final_cka": 0.975068\n   },\n   "123": {\n    "final_accuracy'
    '": 0.9531,\n    "final_cka": 0.976068\n   },\n   "456": {\n    "final_accuracy": 0.9495,\n   '
    ' "final_cka": 0.976939\n   }\n  },\n  "vanilla": {\n   "42": {\n    "final_accuracy": 0.95,\n '
    '   "final_cka": 0.936243\n   },\n   "123": {\n    "final_accuracy": 0.9488,\n    "final_cka"'
    ': 0.940239\n   },\n   "456": {\n    "final_accuracy": 0.9493,\n    "final_cka": 0.939613\n   '
    '}\n  }\n },\n "dry_run": {\n  "mean_ce": 2.4340877532958984,\n  "mean_sp_scaled": 99.12078399'
    '658203,\n  "ratio": 40.72194334618676\n },\n "stage_weights": {\n  "bi_acc": {\n   "layer1": '
    '0.4660857679822313,\n   "layer2": 0.1730736374508799,\n   "layer3": 0.1899880403212028,\n  '
    ' "layer4": 0.17085255424568596\n  },\n  "bi_rep": {\n   "layer1": 0.4641905774202345,\n   "l'
    'ayer2": 0.1700262780130252,\n   "layer3": 0.18372212618523337,\n   "layer4": 0.18206101838'
    '150698\n  },\n  "uniform": {\n   "layer1": 0.25,\n   "layer2": 0.25,\n   "layer3": 0.25,\n   "'
    'layer4": 0.25\n  }\n },\n "loss_share": {\n  "bi_acc": {\n   "epochs": [\n    1,\n    50,\n    1'
    '00,\n    150,\n    200\n   ],\n   "ce": [\n    1.7881,\n    0.2404,\n    0.0112,\n    0.0048,\n  '
    '  0.0047\n   ],\n   "sp": [\n    13.6279,\n    3.1121,\n    0.7288,\n    0.5157,\n    0.4802\n  '
    ' ],\n   "kd": [\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0\n   ]\n  },\n  "bi_rep": {\n   "e'
    'pochs": [\n    1,\n    50,\n    100,\n    150,\n    200\n   ],\n   "ce": [\n    1.7701,\n    0.24'
    '04,\n    0.0111,\n    0.0049,\n    0.0047\n   ],\n   "sp": [\n    14.23,\n    3.2906,\n    0.761'
    '6,\n    0.5424,\n    0.5051\n   ],\n   "kd": [\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0\n '
    '  ]\n  },\n  "uniform": {\n   "epochs": [\n    1,\n    50,\n    100,\n    150,\n    200\n   ],\n  '
    ' "ce": [\n    1.8446,\n    0.2464,\n    0.0128,\n    0.0052,\n    0.005\n   ],\n   "sp": [\n    '
    '19.4399,\n    4.4982,\n    1.0633,\n    0.7345,\n    0.6852\n   ],\n   "kd": [\n    0.0,\n    0.'
    '0,\n    0.0,\n    0.0,\n    0.0\n   ]\n  }\n },\n "fine": {\n  "per_student": {\n   "biacc_seed_1'
    '23": {\n    "cosine_diff_frob": 0.482493,\n    "class_cka_variance": 0.00037831\n   },\n   "'
    'biacc_seed_42": {\n    "cosine_diff_frob": 0.495233,\n    "class_cka_variance": 0.00036483'
    '\n   },\n   "biacc_seed_456": {\n    "cosine_diff_frob": 0.524093,\n    "class_cka_variance"'
    ': 0.00046619\n   },\n   "birep_seed_123": {\n    "cosine_diff_frob": 0.501195,\n    "class_c'
    'ka_variance": 0.00037068\n   },\n   "birep_seed_42": {\n    "cosine_diff_frob": 0.50864,\n  '
    '  "class_cka_variance": 0.00040391\n   },\n   "birep_seed_456": {\n    "cosine_diff_frob": '
    '0.531691,\n    "class_cka_variance": 0.00037062\n   },\n   "uniform_seed_123": {\n    "cosin'
    'e_diff_frob": 0.624664,\n    "class_cka_variance": 0.00071432\n   },\n   "uniform_seed_42":'
    ' {\n    "cosine_diff_frob": 0.621077,\n    "class_cka_variance": 0.00065627\n   },\n   "unif'
    'orm_seed_456": {\n    "cosine_diff_frob": 0.621581,\n    "class_cka_variance": 0.00050114\n'
    '   },\n   "vanilla_seed_123": {\n    "cosine_diff_frob": 1.006565,\n    "class_cka_variance'
    '": 0.00118672\n   },\n   "vanilla_seed_42": {\n    "cosine_diff_frob": 1.283964,\n    "class'
    '_cka_variance": 0.00063802\n   },\n   "vanilla_seed_456": {\n    "cosine_diff_frob": 1.1760'
    '4,\n    "class_cka_variance": 0.00118836\n   }\n  }\n },\n "extended": {\n  "teacher": {\n   "i'
    'cv_mean": 5.3938905715942385,\n   "fdr_critical_pairs": {\n    "cat_dog": 4.98437452316284'
    '2,\n    "cat_deer": 6.116686820983887,\n    "dog_deer": 5.964690208435059\n   },\n   "fdr_me'
    'an_all_pairs": 7.607968351576063,\n   "pr_global": 9.327017784118652,\n   "pr_per_class": '
    '{\n    "0": 8.892683029174805,\n    "1": 8.358110427856445,\n    "2": 9.302284240722656,\n  '
    '  "3": 9.433052062988281,\n    "4": 8.54977798461914,\n    "5": 8.638105392456055,\n    "6"'
    ': 8.93565559387207,\n    "7": 9.953282356262207,\n    "8": 8.658123016357422,\n    "9": 9.5'
    '31828880310059\n   }\n  },\n  "per_student": {\n   "biacc_seed_123": {\n    "icv_mean": 3.565'
    '873,\n    "fdr_critical_pairs": {\n     "cat_dog": 4.306984901428223,\n     "cat_deer": 5.3'
    '221564292907715,\n     "dog_deer": 4.944413185119629\n    },\n    "fdr_mean_all_pairs": 6.2'
    '74876,\n    "pr_global": 9.946342,\n    "pr_per_class": {\n     "0": 10.420859336853027,\n  '
    '   "1": 10.403775215148926,\n     "2": 11.263211250305176,\n     "3": 10.79505443572998,\n '
    '    "4": 9.61878776550293,\n     "5": 9.971114158630371,\n     "6": 10.045051574707031,\n  '
    '   "7": 11.186342239379883,\n     "8": 9.671185493469238,\n     "9": 11.196681022644043\n  '
    '  }\n   },\n   "biacc_seed_42": {\n    "icv_mean": 3.592389,\n    "fdr_critical_pairs": {\n  '
    '   "cat_dog": 4.404469966888428,\n     "cat_deer": 5.3363165855407715,\n     "dog_deer": 4'
    '.980642795562744\n    },\n    "fdr_mean_all_pairs": 6.294852,\n    "pr_global": 9.982198,\n '
    '   "pr_per_class": {\n     "0": 10.277563095092773,\n     "1": 10.542765617370605,\n     "2'
    '": 11.204143524169922,\n     "3": 10.893199920654297,\n     "4": 9.648605346679688,\n     "'
    '5": 10.076518058776855,\n     "6": 10.04422378540039,\n     "7": 11.761462211608887,\n     '
    '"8": 9.87197208404541,\n     "9": 11.165860176086426\n    }\n   },\n   "biacc_seed_456": {\n '
    '   "icv_mean": 3.552407,\n    "fdr_critical_pairs": {\n     "cat_dog": 4.285982608795166,\n'
    '     "cat_deer": 5.483323097229004,\n     "dog_deer": 5.0291876792907715\n    },\n    "fdr_'
    'mean_all_pairs": 6.24516,\n    "pr_global": 9.998255,\n    "pr_per_class": {\n     "0": 10.'
    '66783618927002,\n     "1": 10.622929573059082,\n     "2": 11.266092300415039,\n     "3": 10'
    '.78170394897461,\n     "4": 9.406939506530762,\n     "5": 10.097942352294922,\n     "6": 10'
    '.192303657531738,\n     "7": 11.100605010986328,\n     "8": 9.65811824798584,\n     "9": 11'
    '.061667442321777\n    }\n   },\n   "birep_seed_123": {\n    "icv_mean": 3.585182,\n    "fdr_c'
    'ritical_pairs": {\n     "cat_dog": 4.361828327178955,\n     "cat_deer": 5.386200428009033,'
    '\n     "dog_deer": 5.082632064819336\n    },\n    "fdr_mean_all_pairs": 6.215599,\n    "pr_g'
    'lobal": 9.948627,\n    "pr_per_class": {\n     "0": 10.288797378540039,\n     "1": 10.54223'
    '7281799316,\n     "2": 11.15396499633789,\n     "3": 10.69428825378418,\n     "4": 9.514822'
    '006225586,\n     "5": 9.939373016357422,\n     "6": 10.053226470947266,\n     "7": 11.25628'
    '5667419434,\n     "8": 9.552087783813477,\n     "9": 11.053947448730469\n    }\n   },\n   "bi'
    'rep_seed_42": {\n    "icv_mean": 3.601546,\n    "fdr_critical_pairs": {\n     "cat_dog": 4.'
    '2322916984558105,\n     "cat_deer": 5.329680919647217,\n     "dog_deer": 4.866133689880371'
    '\n    },\n    "fdr_mean_all_pairs": 6.291025,\n    "pr_global": 9.956357,\n    "pr_per_class'
    '": {\n     "0": 10.367018699645996,\n     "1": 10.772022247314453,\n     "2": 11.2852125167'
    '84668,\n     "3": 10.758171081542969,\n     "4": 9.430617332458496,\n     "5": 10.149277687'
    '072754,\n     "6": 10.275106430053711,\n     "7": 11.169583320617676,\n     "8": 9.48492527'
    '0080566,\n     "9": 11.108369827270508\n    }\n   },\n   "birep_seed_456": {\n    "icv_mean":'
    ' 3.611276,\n    "fdr_critical_pairs": {\n     "cat_dog": 4.334062576293945,\n     "cat_deer'
    '": 5.2900309562683105,\n     "dog_deer": 5.129862308502197\n    },\n    "fdr_mean_all_pairs'
    '": 6.220143,\n    "pr_global": 9.959348,\n    "pr_per_class": {\n     "0": 10.5972824096679'
    '69,\n     "1": 10.85382080078125,\n     "2": 10.994085311889648,\n     "3": 10.667876243591'
    '309,\n     "4": 9.219786643981934,\n     "5": 10.212859153747559,\n     "6": 9.901812553405'
    '762,\n     "7": 11.57809829711914,\n     "8": 9.50674819946289,\n     "9": 11.6293210983276'
    '37\n    }\n   },\n   "uniform_seed_123": {\n    "icv_mean": 3.837182,\n    "fdr_critical_pair'
    's": {\n     "cat_dog": 4.293612957000732,\n     "cat_deer": 5.1958088874816895,\n     "dog_'
    'deer": 4.862888336181641\n    },\n    "fdr_mean_all_pairs": 5.951758,\n    "pr_global": 10.'
    '019629,\n    "pr_per_class": {\n     "0": 10.44986629486084,\n     "1": 10.158050537109375,'
    '\n     "2": 11.368271827697754,\n     "3": 10.480777740478516,\n     "4": 9.152318954467773'
    ',\n     "5": 9.721944808959961,\n     "6": 9.671477317810059,\n     "7": 11.3788423538208,\n'
    '     "8": 9.584861755371094,\n     "9": 11.015213966369629\n    }\n   },\n   "uniform_seed_4'
    '2": {\n    "icv_mean": 3.878083,\n    "fdr_critical_pairs": {\n     "cat_dog": 4.2143163681'
    '03027,\n     "cat_deer": 5.075424671173096,\n     "dog_deer": 4.824284553527832\n    },\n   '
    ' "fdr_mean_all_pairs": 5.982457,\n    "pr_global": 10.007527,\n    "pr_per_class": {\n     '
    '"0": 10.296696662902832,\n     "1": 10.158607482910156,\n     "2": 11.175363540649414,\n   '
    '  "3": 11.069903373718262,\n     "4": 9.524827003479004,\n     "5": 10.064114570617676,\n  '
    '   "6": 9.880487442016602,\n     "7": 11.116313934326172,\n     "8": 9.650772094726562,\n  '
    '   "9": 11.027859687805176\n    }\n   },\n   "uniform_seed_456": {\n    "icv_mean": 3.858551'
    ',\n    "fdr_critical_pairs": {\n     "cat_dog": 4.077322006225586,\n     "cat_deer": 5.2038'
    '397789001465,\n     "dog_deer": 4.873816967010498\n    },\n    "fdr_mean_all_pairs": 5.9492'
    '21,\n    "pr_global": 10.030593,\n    "pr_per_class": {\n     "0": 10.436639785766602,\n    '
    ' "1": 10.575350761413574,\n     "2": 11.135380744934082,\n     "3": 10.822219848632812,\n  '
    '   "4": 9.625330924987793,\n     "5": 9.830931663513184,\n     "6": 9.867412567138672,\n   '
    '  "7": 11.340688705444336,\n     "8": 9.679444313049316,\n     "9": 11.390856742858887\n   '
    ' }\n   },\n   "vanilla_seed_123": {\n    "icv_mean": 2.123056,\n    "fdr_critical_pairs": {\n'
    '     "cat_dog": 9.732006072998047,\n     "cat_deer": 12.00318431854248,\n     "dog_deer": '
    '11.330750465393066\n    },\n    "fdr_mean_all_pairs": 14.789433,\n    "pr_global": 9.198489'
    ',\n    "pr_per_class": {\n     "0": 8.408997535705566,\n     "1": 8.191131591796875,\n     "'
    '2": 9.152381896972656,\n     "3": 9.101658821105957,\n     "4": 7.959479331970215,\n     "5'
    '": 7.319938659667969,\n     "6": 7.667062759399414,\n     "7": 8.277702331542969,\n     "8"'
    ': 8.212791442871094,\n     "9": 8.550616264343262\n    }\n   },\n   "vanilla_seed_42": {\n   '
    ' "icv_mean": 1.689695,\n    "fdr_critical_pairs": {\n     "cat_dog": 10.709980010986328,\n '
    '    "cat_deer": 11.877184867858887,\n     "dog_deer": 12.04167366027832\n    },\n    "fdr_m'
    'ean_all_pairs": 16.653766,\n    "pr_global": 9.205914,\n    "pr_per_class": {\n     "0": 8.'
    '786765098571777,\n     "1": 7.271377086639404,\n     "2": 9.366962432861328,\n     "3": 9.1'
    '76907539367676,\n     "4": 8.831122398376465,\n     "5": 7.829574108123779,\n     "6": 8.43'
    '0646896362305,\n     "7": 8.875855445861816,\n     "8": 8.406912803649902,\n     "9": 9.242'
    '242813110352\n    }\n   },\n   "vanilla_seed_456": {\n    "icv_mean": 1.791576,\n    "fdr_cri'
    'tical_pairs": {\n     "cat_dog": 10.37700080871582,\n     "cat_deer": 12.297407150268555,\n'
    '     "dog_deer": 12.537101745605469\n    },\n    "fdr_mean_all_pairs": 15.670102,\n    "pr_'
    'global": 9.248738,\n    "pr_per_class": {\n     "0": 9.041266441345215,\n     "1": 7.529894'
    '828796387,\n     "2": 9.484015464782715,\n     "3": 8.247322082519531,\n     "4": 9.0707283'
    '02001953,\n     "5": 6.838216781616211,\n     "6": 8.474063873291016,\n     "7": 9.28523635'
    '8642578,\n     "8": 8.1803560256958,\n     "9": 9.191299438476562\n    }\n   }\n  }\n },\n "con'
    'fidence": {\n  "teacher": {\n   "accuracy": 0.946399986743927,\n   "mean_confidence_correct'
    '": 0.9896249771118164,\n   "mean_entropy_overall": 0.05021416023373604,\n   "mean_logit_ma'
    'rgin": 9.923263549804688,\n   "calibration_gap": 0.043224990367889404,\n   "critical_pairs'
    '": {\n    "cat_as_dog": {\n     "confidence": 0.899437427520752,\n     "count": 64\n    },\n '
    '   "dog_as_cat": {\n     "confidence": 0.8710804581642151,\n     "count": 56\n    },\n    "c'
    'at_as_deer": {\n     "confidence": 0.8496180176734924,\n     "count": 15\n    },\n    "deer_'
    'as_cat": {\n     "confidence": 0.788044273853302,\n     "count": 13\n    },\n    "dog_as_dee'
    'r": {\n     "confidence": 0.7470632791519165,\n     "count": 14\n    },\n    "deer_as_dog": '
    '{\n     "confidence": 0.8197407722473145,\n     "count": 6\n    }\n   }\n  },\n  "per_student"'
    ': {\n   "biacc_seed_123": {\n    "accuracy": 0.9528,\n    "mean_confidence_correct": 0.9872'
    '45,\n    "mean_entropy_overall": 0.067581,\n    "mean_logit_margin": 8.313264,\n    "calibr'
    'ation_gap": 0.034445,\n    "critical_pairs": {\n     "cat_as_dog": {\n      "confidence": 0'
    '.8751303553581238,\n      "count": 52\n     },\n     "dog_as_cat": {\n      "confidence": 0.'
    '8612459301948547,\n      "count": 51\n     },\n     "cat_as_deer": {\n      "confidence": 0.'
    '8279818296432495,\n      "count": 14\n     },\n     "deer_as_cat": {\n      "confidence": 0.'
    '8490394353866577,\n      "count": 11\n     },\n     "dog_as_deer": {\n      "confidence": 0.'
    '888300895690918,\n      "count": 8\n     },\n     "deer_as_dog": {\n      "confidence": 0.78'
    '40202450752258,\n      "count": 6\n     }\n    }\n   },\n   "biacc_seed_42": {\n    "accuracy"'
    ': 0.9533,\n    "mean_confidence_correct": 0.987227,\n    "mean_entropy_overall": 0.067282,'
    '\n    "mean_logit_margin": 8.318793,\n    "calibration_gap": 0.033927,\n    "critical_pairs'
    '": {\n     "cat_as_dog": {\n      "confidence": 0.876793384552002,\n      "count": 56\n     '
    '},\n     "dog_as_cat": {\n      "confidence": 0.875385582447052,\n      "count": 51\n     },'
    '\n     "cat_as_deer": {\n      "confidence": 0.809459924697876,\n      "count": 11\n     },\n'
    '     "deer_as_cat": {\n      "confidence": 0.8166031837463379,\n      "count": 10\n     },\n'
    '     "dog_as_deer": {\n      "confidence": 0.8080965280532837,\n      "count": 11\n     },\n'
    '     "deer_as_dog": {\n      "confidence": 0.7873353362083435,\n      "count": 7\n     }\n  '
    '  }\n   },\n   "biacc_seed_456": {\n    "accuracy": 0.9505,\n    "mean_confidence_correct": '
    '0.988289,\n    "mean_entropy_overall": 0.06591,\n    "mean_logit_margin": 8.317341,\n    "c'
    'alibration_gap": 0.037789,\n    "critical_pairs": {\n     "cat_as_dog": {\n      "confidenc'
    'e": 0.8628182411193848,\n      "count": 57\n     },\n     "dog_as_cat": {\n      "confidence'
    '": 0.8732329607009888,\n      "count": 49\n     },\n     "cat_as_deer": {\n      "confidence'
    '": 0.8516845703125,\n      "count": 12\n     },\n     "deer_as_cat": {\n      "confidence": '
    '0.7754195332527161,\n      "count": 13\n     },\n     "dog_as_deer": {\n      "confidence": '
    '0.8239200711250305,\n      "count": 12\n     },\n     "deer_as_dog": {\n      "confidence": '
    '0.8667535781860352,\n      "count": 5\n     }\n    }\n   },\n   "birep_seed_123": {\n    "accu'
    'racy": 0.9541,\n    "mean_confidence_correct": 0.986253,\n    "mean_entropy_overall": 0.06'
    '784,\n    "mean_logit_margin": 8.268675,\n    "calibration_gap": 0.032153,\n    "critical_p'
    'airs": {\n     "cat_as_dog": {\n      "confidence": 0.8539800047874451,\n      "count": 57\n'
    '     },\n     "dog_as_cat": {\n      "confidence": 0.8770161271095276,\n      "count": 50\n '
    '    },\n     "cat_as_deer": {\n      "confidence": 0.8136473894119263,\n      "count": 13\n '
    '    },\n     "deer_as_cat": {\n      "confidence": 0.8384568691253662,\n      "count": 9\n  '
    '   },\n     "dog_as_deer": {\n      "confidence": 0.7904630303382874,\n      "count": 12\n  '
    '   },\n     "deer_as_dog": {\n      "confidence": 0.8879022598266602,\n      "count": 5\n   '
    '  }\n    }\n   },\n   "birep_seed_42": {\n    "accuracy": 0.9527,\n    "mean_confidence_corre'
    'ct": 0.987439,\n    "mean_entropy_overall": 0.065707,\n    "mean_logit_margin": 8.294335,\n'
    '    "calibration_gap": 0.034739,\n    "critical_pairs": {\n     "cat_as_dog": {\n      "con'
    'fidence": 0.8851126432418823,\n      "count": 57\n     },\n     "dog_as_cat": {\n      "conf'
    'idence": 0.8870309591293335,\n      "count": 52\n     },\n     "cat_as_deer": {\n      "conf'
    'idence": 0.8286765813827515,\n      "count": 10\n     },\n     "deer_as_cat": {\n      "conf'
    'idence": 0.8899716734886169,\n      "count": 11\n     },\n     "dog_as_deer": {\n      "conf'
    'idence": 0.91837477684021,\n      "count": 9\n     },\n     "deer_as_dog": {\n      "confide'
    'nce": 0.760709285736084,\n      "count": 7\n     }\n    }\n   },\n   "birep_seed_456": {\n    '
    '"accuracy": 0.9494,\n    "mean_confidence_correct": 0.986989,\n    "mean_entropy_overall":'
    ' 0.06794,\n    "mean_logit_margin": 8.351665,\n    "calibration_gap": 0.037589,\n    "criti'
    'cal_pairs": {\n     "cat_as_dog": {\n      "confidence": 0.8641697764396667,\n      "count"'
    ': 60\n     },\n     "dog_as_cat": {\n      "confidence": 0.8864068388938904,\n      "count":'
    ' 56\n     },\n     "cat_as_deer": {\n      "confidence": 0.7957964539527893,\n      "count":'
    ' 14\n     },\n     "deer_as_cat": {\n      "confidence": 0.7781854867935181,\n      "count":'
    ' 14\n     },\n     "dog_as_deer": {\n      "confidence": 0.8801409006118774,\n      "count":'
    ' 15\n     },\n     "deer_as_dog": {\n      "confidence": 0.9199509620666504,\n      "count":'
    ' 3\n     }\n    }\n   },\n   "uniform_seed_123": {\n    "accuracy": 0.9531,\n    "mean_confide'
    'nce_correct": 0.986238,\n    "mean_entropy_overall": 0.071031,\n    "mean_logit_margin": 8'
    '.284298,\n    "calibration_gap": 0.033138,\n    "critical_pairs": {\n     "cat_as_dog": {\n '
    '     "confidence": 0.8665250539779663,\n      "count": 55\n     },\n     "dog_as_cat": {\n  '
    '    "confidence": 0.8489180207252502,\n      "count": 60\n     },\n     "cat_as_deer": {\n  '
    '    "confidence": 0.8479725122451782,\n      "count": 14\n     },\n     "deer_as_cat": {\n  '
    '    "confidence": 0.9047413468360901,\n      "count": 11\n     },\n     "dog_as_deer": {\n  '
    '    "confidence": 0.7538452744483948,\n      "count": 11\n     },\n     "deer_as_dog": {\n  '
    '    "confidence": 0.9796417355537415,\n      "count": 3\n     }\n    }\n   },\n   "uniform_se'
    'ed_42": {\n    "accuracy": 0.9532,\n    "mean_confidence_correct": 0.987173,\n    "mean_ent'
    'ropy_overall": 0.067905,\n    "mean_logit_margin": 8.338542,\n    "calibration_gap": 0.033'
    '973,\n    "critical_pairs": {\n     "cat_as_dog": {\n      "confidence": 0.8715521693229675'
    ',\n      "count": 57\n     },\n     "dog_as_cat": {\n      "confidence": 0.8762630820274353,'
    '\n      "count": 52\n     },\n     "cat_as_deer": {\n      "confidence": 0.714372992515564,\n'
    '      "count": 11\n     },\n     "deer_as_cat": {\n      "confidence": 0.8460721373558044,\n'
    '      "count": 11\n     },\n     "dog_as_deer": {\n      "confidence": 0.9251725673675537,\n'
    '      "count": 9\n     },\n     "deer_as_dog": {\n      "confidence": 0.8483384251594543,\n '
    '     "count": 5\n     }\n    }\n   },\n   "uniform_seed_456": {\n    "accuracy": 0.9495,\n    '
    '"mean_confidence_correct": 0.987467,\n    "mean_entropy_overall": 0.067944,\n    "mean_log'
    'it_margin": 8.319497,\n    "calibration_gap": 0.037967,\n    "critical_pairs": {\n     "cat'
    '_as_dog": {\n      "confidence": 0.8881133794784546,\n      "count": 58\n     },\n     "dog_'
    'as_cat": {\n      "confidence": 0.8229732513427734,\n      "count": 60\n     },\n     "cat_a'
    's_deer": {\n      "confidence": 0.8166742920875549,\n      "count": 12\n     },\n     "deer_'
    'as_cat": {\n      "confidence": 0.8000280261039734,\n      "count": 11\n     },\n     "dog_a'
    's_deer": {\n      "confidence": 0.7621651291847229,\n      "count": 13\n     },\n     "deer_'
    'as_dog": {\n      "confidence": 0.8860183358192444,\n      "count": 4\n     }\n    }\n   },\n '
    '  "vanilla_seed_123": {\n    "accuracy": 0.9496,\n    "mean_confidence_correct": 0.988906,'
    '\n    "mean_entropy_overall": 0.057343,\n    "mean_logit_margin": 8.403078,\n    "calibrati'
    'on_gap": 0.039306,\n    "critical_pairs": {\n     "cat_as_dog": {\n      "confidence": 0.88'
    '31296563148499,\n      "count": 63\n     },\n     "dog_as_cat": {\n      "confidence": 0.876'
    '4033913612366,\n      "count": 46\n     },\n     "cat_as_deer": {\n      "confidence": 0.840'
    '4882550239563,\n      "count": 10\n     },\n     "deer_as_cat": {\n      "confidence": 0.790'
    '6137108802795,\n      "count": 7\n     },\n     "dog_as_deer": {\n      "confidence": 0.8808'
    '94660949707,\n      "count": 6\n     },\n     "deer_as_dog": {\n      "confidence": 0.706744'
    '7900772095,\n      "count": 10\n     }\n    }\n   },\n   "vanilla_seed_42": {\n    "accuracy":'
    ' 0.9499,\n    "mean_confidence_correct": 0.988397,\n    "mean_entropy_overall": 0.061986,\n'
    '    "mean_logit_margin": 8.241878,\n    "calibration_gap": 0.038497,\n    "critical_pairs"'
    ': {\n     "cat_as_dog": {\n      "confidence": 0.8537514209747314,\n      "count": 58\n     '
    '},\n     "dog_as_cat": {\n      "confidence": 0.8574411273002625,\n      "count": 56\n     }'
    ',\n     "cat_as_deer": {\n      "confidence": 0.8139582872390747,\n      "count": 9\n     },'
    '\n     "deer_as_cat": {\n      "confidence": 0.8136517405509949,\n      "count": 12\n     },'
    '\n     "dog_as_deer": {\n      "confidence": 0.8238508105278015,\n      "count": 14\n     },'
    '\n     "deer_as_dog": {\n      "confidence": 0.8991637825965881,\n      "count": 3\n     }\n '
    '   }\n   },\n   "vanilla_seed_456": {\n    "accuracy": 0.9514,\n    "mean_confidence_correct'
    '": 0.988058,\n    "mean_entropy_overall": 0.063018,\n    "mean_logit_margin": 8.188416,\n  '
    '  "calibration_gap": 0.036658,\n    "critical_pairs": {\n     "cat_as_dog": {\n      "confi'
    'dence": 0.8216279745101929,\n      "count": 59\n     },\n     "dog_as_cat": {\n      "confid'
    'ence": 0.8332595825195312,\n      "count": 44\n     },\n     "cat_as_deer": {\n      "confid'
    'ence": 0.7724066376686096,\n      "count": 15\n     },\n     "deer_as_cat": {\n      "confid'
    'ence": 0.8977485299110413,\n      "count": 7\n     },\n     "dog_as_deer": {\n      "confide'
    'nce": 0.7055860161781311,\n      "count": 11\n     },\n     "deer_as_dog": {\n      "confide'
    'nce": 0.7151951789855957,\n      "count": 2\n     }\n    }\n   }\n  }\n },\n "cifar10c": {\n  "c'
    'orruption_types": [\n   "gaussian_noise",\n   "shot_noise",\n   "impulse_noise",\n   "defocu'
    's_blur",\n   "glass_blur",\n   "motion_blur",\n   "zoom_blur",\n   "snow",\n   "frost",\n   "f'
    'og",\n   "brightness",\n   "contrast",\n   "elastic_transform",\n   "pixelate",\n   "jpeg_com'
    'pression",\n   "speckle_noise",\n   "gaussian_blur",\n   "spatter",\n   "saturate"\n  ],\n  "a'
    'ggregate": {\n   "bi_acc": {\n    "mCE_mean": 0.9598070200524943,\n    "mCE_std": 0.0042740'
    '05603869726,\n    "rmCE_mean": 0.9896484796575726,\n    "rmCE_std": 0.006400624375513853,\n'
    '    "mean_ece_clean": 0.026640179338554542,\n    "mean_ece_clean_std": 0.0014353363593446'
    '668,\n    "mean_ece_corrupted": 0.17688117636773146,\n    "mean_ece_corrupted_std": 0.0031'
    '669756029030043\n   },\n   "bi_rep": {\n    "mCE_mean": 0.9591836806121664,\n    "mCE_std": '
    '0.00981216532652022,\n    "rmCE_mean": 0.9891755251441013,\n    "rmCE_std": 0.015986623285'
    '21778,\n    "mean_ece_clean": 0.02752719782193502,\n    "mean_ece_clean_std": 0.0009549602'
    '62835739,\n    "mean_ece_corrupted": 0.17664952372207057,\n    "mean_ece_corrupted_std": 0'
    '.0019392408507383485\n   },\n   "uniform": {\n    "mCE_mean": 0.9654957876811019,\n    "mCE_'
    'std": 0.006912375096771576,\n    "rmCE_mean": 0.9956902598306923,\n    "rmCE_std": 0.00598'
    '5456441436744,\n    "mean_ece_clean": 0.026842966698110104,\n    "mean_ece_clean_std": 0.0'
    '015428820278835247,\n    "mean_ece_corrupted": 0.17749266371983316,\n    "mean_ece_corrupt'
    'ed_std": 0.0015699177297843793\n   },\n   "vanilla": {\n    "mCE_mean": 0.9481694813453855,'
    '\n    "mCE_std": 0.012062092791348145,\n    "rmCE_mean": 0.9592208192074664,\n    "rmCE_std'
    '": 0.023088178372917028,\n    "mean_ece_clean": 0.029740246586501595,\n    "mean_ece_clean'
    '_std": 0.001405280168749687,\n    "mean_ece_corrupted": 0.18178373504395942,\n    "mean_ec'
    'e_corrupted_std": 0.003968120095989646\n   }\n  },\n  "teacher_clean_accuracy": 0.946399986'
    '743927,\n  "teacher_clean_ece": 0.034949187648296354,\n  "ece_per_corruption": {\n   "cond_'
    'bi_acc": {\n    "gaussian_noise": 0.41846,\n    "shot_noise": 0.306126,\n    "impulse_noise'
    '": 0.336981,\n    "defocus_blur": 0.126483,\n    "glass_blur": 0.331898,\n    "motion_blur"'
    ': 0.155635,\n    "zoom_blur": 0.153376,\n    "snow": 0.11031,\n    "frost": 0.153664,\n    "'
    'fog": 0.079852,\n    "brightness": 0.035796,\n    "contrast": 0.148931,\n    "elastic_trans'
    'form": 0.096489,\n    "pixelate": 0.172889,\n    "jpeg_compression": 0.119426,\n    "speckl'
    'e_noise": 0.273348,\n    "gaussian_blur": 0.20743,\n    "spatter": 0.085314,\n    "saturate'
    '": 0.048334\n   },\n   "cond_bi_rep": {\n    "gaussian_noise": 0.414322,\n    "shot_noise": '
    '0.304909,\n    "impulse_noise": 0.324065,\n    "defocus_blur": 0.127746,\n    "glass_blur":'
    ' 0.333294,\n    "motion_blur": 0.153692,\n    "zoom_blur": 0.153146,\n    "snow": 0.110336,'
    '\n    "frost": 0.154461,\n    "fog": 0.080024,\n    "brightness": 0.036383,\n    "contrast":'
    ' 0.151271,\n    "elastic_transform": 0.096529,\n    "pixelate": 0.174738,\n    "jpeg_compre'
    'ssion": 0.121009,\n    "speckle_noise": 0.27349,\n    "gaussian_blur": 0.213548,\n    "spat'
    'ter": 0.08482,\n    "saturate": 0.048556\n   },\n   "cond_uniform": {\n    "gaussian_noise":'
    ' 0.414818,\n    "shot_noise": 0.302026,\n    "impulse_noise": 0.343582,\n    "defocus_blur"'
    ': 0.130362,\n    "glass_blur": 0.331131,\n    "motion_blur": 0.156437,\n    "zoom_blur": 0.'
    '156885,\n    "snow": 0.108678,\n    "frost": 0.154217,\n    "fog": 0.07988,\n    "brightness'
    '": 0.036199,\n    "contrast": 0.149325,\n    "elastic_transform": 0.096388,\n    "pixelate"'
    ': 0.170089,\n    "jpeg_compression": 0.117014,\n    "speckle_noise": 0.269118,\n    "gaussi'
    'an_blur": 0.22268,\n    "spatter": 0.085601,\n    "saturate": 0.047932\n   },\n   "cond_vani'
    'lla": {\n    "gaussian_noise": 0.462353,\n    "shot_noise": 0.339877,\n    "impulse_noise":'
    ' 0.34163,\n    "defocus_blur": 0.114523,\n    "glass_blur": 0.337364,\n    "motion_blur": 0'
    '.149482,\n    "zoom_blur": 0.141298,\n    "snow": 0.113642,\n    "frost": 0.146307,\n    "fo'
    'g": 0.075771,\n    "brightness": 0.038142,\n    "contrast": 0.150734,\n    "elastic_transfo'
    'rm": 0.097946,\n    "pixelate": 0.177296,\n    "jpeg_compression": 0.135762,\n    "speckle_'
    'noise": 0.296206,\n    "gaussian_blur": 0.183015,\n    "spatter": 0.103674,\n    "saturate"'
    ': 0.048869\n   },\n   "teacher": {\n    "gaussian_noise": 0.471594,\n    "shot_noise": 0.352'
    '363,\n    "impulse_noise": 0.334471,\n    "defocus_blur": 0.153254,\n    "glass_blur": 0.38'
    '099,\n    "motion_blur": 0.180194,\n    "zoom_blur": 0.18443,\n    "snow": 0.126019,\n    "f'
    'rost": 0.168479,\n    "fog": 0.090598,\n    "brightness": 0.043573,\n    "contrast": 0.1666'
    '14,\n    "elastic_transform": 0.120503,\n    "pixelate": 0.199778,\n    "jpeg_compression":'
    ' 0.141069,\n    "speckle_noise": 0.312583,\n    "gaussian_blur": 0.249651,\n    "spatter": '
    '0.09769,\n    "saturate": 0.05963\n   }\n  }\n },\n "cifar10p": {\n  "aggregate": {\n   "bi_acc'
    '": {\n    "mFP_mean": 0.033019611159648375,\n    "mFP_std": 0.0003190967335293082,\n    "mF'
    'PR_mean": 0.8573400005577821,\n    "mFPR_std": 0.008391221160004356\n   },\n   "bi_rep": {\n'
    '    "mFP_mean": 0.0329039444895251,\n    "mFP_std": 0.0005633204919708969,\n    "mFPR_mean'
    '": 0.8497241240356201,\n    "mFPR_std": 0.01623107549341456\n   },\n   "uniform": {\n    "mF'
    'P_mean": 0.03304172230810865,\n    "mFP_std": 0.0003645760676208341,\n    "mFPR_mean": 0.8'
    '53645221189509,\n    "mFPR_std": 0.010084018621257426\n   },\n   "vanilla": {\n    "mFP_mean'
    '": 0.03398788888990465,\n    "mFP_std": 0.00042793456712774,\n    "mFPR_mean": 0.898571294'
    '6795049,\n    "mFPR_std": 0.007368382794255481\n   }\n  },\n  "teacher_mFP": 0.0383575001647'
    '3564\n },\n "provenance": {\n  "source": "Bi_project/phase4_results/*.json \\u2014 archived '
    'artifacts of the 12 primary Phase-4 training runs (4 conditions \\u00d7 3 seeds, 200 epoc'
    'hs, RTX 4070 + Kaggle)",\n  "training_setup": "ResNet-50 teacher (edadaltocg/resnet50_cif'
    'ar10, frozen, acc 0.9464); ResNet-18 CIFAR-stem students; SGD Nesterov lr 0.1, wd 5e-4, '
    'batch 128, 200 epochs, step decay x0.1 @ {60,120,160}; gamma=3000; seeds {42,123,456}",\n'
    '  "note": "Values are read-only snapshots embedded so the Phase-4 figures/tables of the '
    'report are reproducible without re-training; full from-scratch re-training and post-hoc '
    'recomputation code is included in the notebook behind RUN flags."\n }\n}'
)

ARCHIVED = json.loads(ARCHIVED_PHASE4_JSON)
CONDITIONS_P4 = ["bi_acc", "bi_rep", "uniform", "vanilla"]
COND_LABELS = {"bi_acc": "BI-Acc", "bi_rep": "BI-Rep",
               "uniform": "Uniform", "vanilla": "Vanilla"}
COND_COLORS = {"bi_acc": "#DD8452", "bi_rep": "#55A868",
               "uniform": "#4C72B0", "vanilla": "#C44E52"}
TEACHER_COLOR = "#8172B2"
COND_NORM = {"biacc": "bi_acc", "birep": "bi_rep",
             "uniform": "uniform", "vanilla": "vanilla"}

def group_per_student(per_student):
    grouped = {c: [] for c in CONDITIONS_P4}
    for key, entry in per_student.items():
        prefix = key.rsplit("_seed_", 1)[0]
        cond = COND_NORM.get(prefix)
        if cond:
            grouped[cond].append(entry)
    return grouped

def cond_mean_std(grouped, fn):
    return {c: (float(np.mean([fn(e) for e in entries])) if entries else 0.0,
                float(np.std([fn(e) for e in entries])) if entries else 0.0)
            for c, entries in grouped.items()}

print("Embedded artifact snapshot:",
      len(ARCHIVED_PHASE4_JSON) // 1024, "KB —",
      ", ".join(sorted(ARCHIVED.keys())))

In [ ]:
# ── Cause 1 (live): stage-averaging collapses bi_acc vs bi_rep weights ─────
condition_weights = build_condition_weights(phase2_results)

print("BI-derived stage weights computed from THIS session's Phase-2 metrics:\n")
wtab = pd.DataFrame({c: condition_weights[c] for c in ("uniform", "bi_acc", "bi_rep")}).T
display(wtab.round(4))

wa = np.array([condition_weights["bi_acc"][s] for s in STAGES])
wr = np.array([condition_weights["bi_rep"][s] for s in STAGES])
max_diff_pp = 100 * np.abs(wa - wr).max()
cos_wa_wr = float(np.dot(wa, wr) / (np.linalg.norm(wa) * np.linalg.norm(wr)))
pearson_r = float(stats.pearsonr(wa, wr).statistic)

print(f"max |bi_acc − bi_rep| over stages : {max_diff_pp:.2f} percentage points "
      f"(paper: 1.12 pp)")
print(f"cosine similarity                 : {cos_wa_wr:.4f}   (paper: 0.9997)")
print(f"Pearson r                         : {pearson_r:.4f}   (paper: 0.9986)")
print("\n→ The two conditions were near-duplicates BEFORE training began:"
      "\n  layer4.0's Δ — the project's central finding — contributes one third of ONE"
      "\n  stage mean; six near-zero-Δ blocks dilute it away (low-pass filter).")

# Cross-check against archived training-time weight vectors
arch_bi_rep_w = ARCHIVED["stage_weights"]["bi_rep"]
print("\nArchived bi_rep weights used by the original runs:",
      {k: round(v, 4) for k, v in arch_bi_rep_w.items()})
assert all(abs(condition_weights["bi_rep"][s] - arch_bi_rep_w[s]) < 0.02
           for s in STAGES), "stage weights deviate from archived runs"

In [ ]:
# ── Cause 2 (live): γ = 3000 makes any normalised weighting inert — dry run ─
@torch.no_grad()
def dry_run(student, teacher_mod, loader, device, weights, gamma, n_batches=5):
    """Measure CE vs γ·SP at initialisation (the guard that SHOULD have stopped γ=3000)."""
    teacher_mod.eval(); student.train()
    criterion = nn.CrossEntropyLoss()
    hooks = SPHookManager(teacher_mod, student)
    ces, sps = [], []
    it = iter(loader)
    for _ in range(n_batches):
        try:
            images, labels = next(it)
        except StopIteration:
            break
        images, labels = images.to(device), labels.to(device)
        teacher_mod(images)                       # fills teacher feats
        logits = student(images)
        ces.append(criterion(logits, labels).item())
        sps.append((gamma * sp_kd_loss(hooks.teacher_feats, hooks.student_feats,
                                       weights)).item())
    hooks.remove(); student.zero_grad()
    mean_ce = float(np.mean(ces)); mean_sp = float(np.mean(sps))
    return {"mean_ce": round(mean_ce, 4), "mean_sp_scaled": round(mean_sp, 4),
            "ratio": round(mean_sp / (mean_ce + 1e-8), 2)}


train_ds_aug = datasets.CIFAR10(str(DATA_DIR), train=True, download=False,
                                transform=get_train_transform())
aug_loader = DataLoader(train_ds_aug, batch_size=PHASE4_BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        worker_init_fn=worker_init_fn)

set_seed(SEED)
dry_student = build_student().to(DEVICE)
dry = dry_run(dry_student, teacher, aug_loader, DEVICE,
              condition_weights["uniform"], PHASE4_GAMMA_KD)
del dry_student
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

arch_dry = ARCHIVED["dry_run"]
print("Dry run (this session) :", dry)
print("Archived dry run       :", arch_dry)
sp_share_now = dry["mean_sp_scaled"] / (dry["mean_sp_scaled"] + dry["mean_ce"])
print(f"\nγ·SP is {sp_share_now*100:.0f}% of the loss at epoch 0 — "
      f"the archived runs reached ~99% by epoch 200 (warning threshold was ratio>100;"
      f" measured 40.7 passed silently). Any normalised stage weighting therefore has"
      f" ≈ zero gradient leverage.")

In [ ]:
# ── Loss composition over training (from archived artifacts) ───────────────
rows = {}
for cond, ls in ARCHIVED["loss_share"].items():
    rows[cond] = {
        f"SP share @ ep {e}":
            f"{ls['sp'][i] / (ls['ce'][i] + ls['sp'][i] + ls['kd'][i] + 1e-9) * 100:.1f}%"
        for i, e in enumerate(ls["epochs"])
    }
print("γ·SP share of total loss across training (seed-mean, archived runs):")
display(pd.DataFrame(rows).T)

fig, ax = plt.subplots(figsize=(8, 4.5))
for cond, ls in ARCHIVED["loss_share"].items():
    shares = [ls["sp"][i] / (ls["ce"][i] + ls["sp"][i] + ls["kd"][i] + 1e-9) * 100
              for i in range(len(ls["epochs"]))]
    ax.plot(ls["epochs"], shares, marker="o", label=cond)
ax.axhline(80, color="grey", ls="--", lw=1, label="80% budget (post-hoc fix trigger)")
ax.set_xscale("log"); ax.set_xlabel("epoch (log scale)")
ax.set_ylabel("γ·SP share of total loss [%]")
ax.set_title("The weighted term dominates the objective — weighting cannot matter")
ax.legend(fontsize=8); fig.tight_layout()
save_fig(fig, "fig_sp_loss_share.png")

### 5.1 · Full re-training code (optional — heavy)

`run_condition` below is the faithful port of the original trainer
(SGD-Nesterov, step decay, gradient clipping for 10 epochs, CKA probe every 20
epochs). Set `RUN_PHASE4_TRAINING=True` to execute — realistically on a subset,
e.g. `PHASE4_CONDITIONS_TO_TRAIN=["vanilla","uniform"]`,
`PHASE4_SEEDS_TO_TRAIN=[42]`. With `ADAPTIVE_GAMMA=False` (default) it reproduces
the archived protocol exactly; the post-hoc γ-halving fix of repo commit
`d2f425a` is included behind the flag for completeness.

In [ ]:
def run_condition(condition, weights, teacher_mod, train_ds_, test_loader_,
                  calib_loader_, device, seed, n_epochs=PHASE4_N_EPOCHS,
                  gamma=PHASE4_GAMMA_KD):
    print(f"  [{condition}] seed={seed}")
    set_seed(seed)
    relax_determinism_for_training()
    student = build_student().to(device)
    train_loader = DataLoader(train_ds_, batch_size=PHASE4_BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              worker_init_fn=worker_init_fn,
                              generator=get_dataloader_generator(seed),
                              persistent_workers=NUM_WORKERS > 0)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(student.parameters(), lr=PHASE4_LR,
                                momentum=PHASE4_MOMENTUM,
                                weight_decay=PHASE4_WEIGHT_DECAY, nesterov=True)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer,
                                                     milestones=PHASE4_LR_MILESTONES,
                                                     gamma=PHASE4_LR_DECAY)
    hooks = SPHookManager(teacher_mod, student)
    teacher_mod.eval()
    acc_curve, ce_curve, sp_curve, cka_curve = [], [], [], []

    use_sp = condition != "vanilla"

    for epoch in range(1, n_epochs + 1):
        t_ep = time.time()
        student.train()
        sum_ce = sum_sp = n_b = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            if use_sp:
                with torch.no_grad():
                    teacher_mod(images)
            optimizer.zero_grad()
            logits = student(images)
            loss_ce = criterion(logits, labels)
            if use_sp:
                loss_sp = sp_kd_loss(hooks.teacher_feats, hooks.student_feats, weights)
                loss = loss_ce + gamma * loss_sp
            else:
                loss_sp = torch.zeros(1, device=device)
                loss = loss_ce
            loss.backward()
            if epoch <= PHASE4_GRAD_CLIP_EPOCHS:
                nn.utils.clip_grad_norm_(student.parameters(), PHASE4_GRAD_CLIP)
            optimizer.step()
            sum_ce += loss_ce.item(); sum_sp += (gamma * loss_sp).item(); n_b += 1
        scheduler.step()

        # Post-hoc fix (repo commit d2f425a): halve γ while SP > 80% of loss @ epoch 10
        if ADAPTIVE_GAMMA and epoch == 10 and use_sp:
            tot = (sum_ce / n_b) + (sum_sp / n_b)
            frac = (sum_sp / n_b) / (tot + 1e-8)
            while frac > 0.80:
                gamma *= 0.5
                frac *= 0.5
            print(f"    adaptive γ @ epoch 10 → {gamma:.0f}")

        if epoch % 5 == 0 or epoch == n_epochs:
            test_acc = evaluate_student(student, test_loader_, device)
        else:
            test_acc = acc_curve[-1] if acc_curve else 0.0
        acc_curve.append(test_acc)
        ce_curve.append(sum_ce / n_b); sp_curve.append(sum_sp / n_b)

        if epoch % PHASE4_CKA_INTERVAL == 0 or epoch == n_epochs:
            cka_val = compute_transfer_cka(student, teacher_mod, calib_loader_, device)
            cka_curve.append([epoch, cka_val])
            print(f"    epoch {epoch:3d}/{n_epochs} | acc={test_acc:.4f} | "
                  f"CE={ce_curve[-1]:.4f} | γ·SP={sp_curve[-1]:.4f} | "
                  f"CKA={cka_val:.4f} | {time.time()-t_ep:.1f}s")

    hooks.remove()
    final_cka = cka_curve[-1][1] if cka_curve else None
    print(f"  [{condition}] seed={seed} done — acc={acc_curve[-1]:.4f} CKA={final_cka:.4f}")
    return {
        "final_accuracy": acc_curve[-1], "final_cka": final_cka,
        "accuracy_curve": acc_curve, "ce_loss_curve": ce_curve,
        "sp_loss_curve": sp_curve, "cka_curve": cka_curve,
        "student_state_dict": student.state_dict(),
    }


def run_phase4_training(conditions=None, seeds=None, n_epochs=None):
    conditions = conditions or PHASE4_CONDITIONS_TO_TRAIN
    seeds = seeds or PHASE4_SEEDS_TO_TRAIN
    n_epochs = n_epochs or PHASE4_N_EPOCHS
    ckpt_root = PHASE4_DIR / "retrained_checkpoints"
    ckpt_root.mkdir(parents=True, exist_ok=True)
    results = {}
    for cond in conditions:
        w = condition_weights[cond]
        print(f"\n{'─'*60}\nCondition: {cond}  weights: "
              f"{ {k: round(v, 4) for k, v in w.items()} }")
        results[cond] = {}
        for seed in seeds:
            run_res = run_condition(cond, w, teacher, train_ds_aug, test_loader,
                                    calib_loader, DEVICE, seed, n_epochs=n_epochs)
            sd = run_res.pop("student_state_dict")
            torch.save(sd, ckpt_root / f"{cond}_student_seed_{seed}.pt")
            results[cond][f"seed_{seed}"] = run_res
    with open(PHASE4_DIR / f"phase4_retrained_n{n_epochs}.json", "w") as f:
        json.dump(results, f, indent=2)
    print("Phase-4 re-training complete →", ckpt_root)
    return results

if RUN_PHASE4_TRAINING:
    banner("PHASE 4 TRAINING (subset selected via flags)")
    live_runs = run_phase4_training()
else:
    print(f"RUN_PHASE4_TRAINING = False — skipping ~{len(PHASE4_SEEDS)*200}+ epochs of training."
          "\nArchived artifacts (next cell) provide the published numbers.")

In [ ]:
# ── Table 2 — final accuracy & teacher–student CKA (+ Welch tests, §5.3) ───
rows = {}
acc_by_cond = {}
for cond, seeds_d in ARCHIVED["final_runs"].items():
    accs = [seeds_d[str(s)]["final_accuracy"] for s in PHASE4_SEEDS]
    ckas = [seeds_d[str(s)]["final_cka"] for s in PHASE4_SEEDS]
    acc_by_cond[cond] = np.array(accs)
    rows[COND_LABELS[cond]] = {
        "acc (42)": accs[0], "acc (123)": accs[1], "acc (456)": accs[2],
        "mean acc": round(float(np.mean(accs)), 4),
        "CKA range": f"{min(ckas):.4f}–{max(ckas):.4f}",
        "mean CKA": round(float(np.mean(ckas)), 4),
    }
display(pd.DataFrame(rows).T)

def welch(a, b):
    t, p = stats.ttest_ind(acc_by_cond[a], acc_by_cond[b], equal_var=False)
    return round(float(p), 3)

print("Pairwise Welch t-tests on accuracy (paper §5.3: 0.97 / 0.67 / 0.76; "
      "vs vanilla 0.07–0.19):")
for a, b in [("bi_acc", "bi_rep"), ("bi_acc", "uniform"), ("bi_rep", "uniform"),
             ("uniform", "vanilla"), ("bi_acc", "vanilla"), ("bi_rep", "vanilla")]:
    print(f"  p({COND_LABELS[a]} vs {COND_LABELS[b]}) = {welch(a, b):.3f}")

In [ ]:
# ── Figure P4-1 — aggregate representation fidelity per condition ──────────
def fig1_aggregate_bar(fine_per_student, title_suffix="(archived runs)"):
    grouped = group_per_student(fine_per_student)
    frob_ms = cond_mean_std(grouped, lambda e: e["cosine_diff_frob"])
    var_ms = cond_mean_std(grouped, lambda e: e["class_cka_variance"])
    x = np.arange(len(CONDITIONS_P4)); w = .35
    fig, ax1 = plt.subplots(figsize=(8, 5)); ax2 = ax1.twinx()
    ax1.bar(x - w/2, [frob_ms[c][0] for c in CONDITIONS_P4], w,
            yerr=[frob_ms[c][1] for c in CONDITIONS_P4], capsize=4, alpha=.85,
            color=[COND_COLORS[c] for c in CONDITIONS_P4],
            label="Cosine Frobenius (left)")
    ax2.bar(x + w/2, [var_ms[c][0] for c in CONDITIONS_P4], w,
            yerr=[var_ms[c][1] for c in CONDITIONS_P4], capsize=4, alpha=.45,
            hatch="//", color=[COND_COLORS[c] for c in CONDITIONS_P4],
            label="Class-CKA variance (right)")
    ax1.set_xticks(x); ax1.set_xticklabels([COND_LABELS[c] for c in CONDITIONS_P4])
    ax1.set_ylabel("Cosine Frobenius norm (dev from teacher)")
    ax2.set_ylabel("Per-class CKA variance")
    ax1.set_title(f"Fidelity to teacher geometry by condition {title_suffix}")
    h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, fontsize=9)
    fig.tight_layout(); save_fig(fig, "fig1_aggregate_bar.png")

fig1_aggregate_bar(ARCHIVED["fine"]["per_student"])

In [ ]:
# ── Figures P4-7 / P4-8 / P4-9 — ICV, FDR, Participation Ratio ─────────────
def fig7_intra_class_variance(ext):
    teacher_icv = ext["teacher"]["icv_mean"]
    grouped = group_per_student(ext["per_student"])
    stats_ = cond_mean_std(grouped, lambda e: e["icv_mean"])
    labels = ["Teacher"] + [COND_LABELS[c] for c in CONDITIONS_P4]
    means = [teacher_icv] + [stats_[c][0] for c in CONDITIONS_P4]
    errs = [0.0] + [stats_[c][1] for c in CONDITIONS_P4]
    colors = [TEACHER_COLOR] + [COND_COLORS[c] for c in CONDITIONS_P4]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(np.arange(len(labels)), means, yerr=errs, capsize=5,
                  color=colors, alpha=.85)
    bars[0].set_hatch("//")
    pad = max(means) * .01
    for xi, (val, err) in enumerate(zip(means, errs)):
        ax.text(xi, val + err + pad, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(np.arange(len(labels))); ax.set_xticklabels(labels)
    ax.set_ylabel("Mean intra-class variance")
    ax.set_title("Intra-class variance — vanilla collapses clusters ~3× tighter "
                 "than the teacher\n(SP-KD students inherit the teacher's dispersion)",
                 fontsize=11)
    fig.tight_layout(); save_fig(fig, "fig7_intra_class_variance.png")


def fig8_fdr_critical_pairs(ext):
    pairs = ["cat_dog", "cat_deer", "dog_deer"]
    pair_labels = ["Cat–Dog", "Cat–Deer", "Dog–Deer", "All-pairs mean"]
    n_groups, n_bars, width = 4, 5, .14
    x = np.arange(n_groups)
    teacher = ext["teacher"]
    grouped = group_per_student(ext["per_student"])

    def cond_stats(cond):
        res = [(float(np.mean(vs)), float(np.std(vs)))
               for vs in ([e["fdr_critical_pairs"].get(p, 0.) for e in grouped[cond]]
                          for p in pairs)]
        allv = [e["fdr_mean_all_pairs"] for e in grouped[cond]]
        res.append((float(np.mean(allv)), float(np.std(allv))))
        return res

    bars_def = [([teacher["fdr_critical_pairs"].get(p, 0.) for p in pairs]
                 + [teacher["fdr_mean_all_pairs"]], [0.] * n_groups,
                 TEACHER_COLOR, "Teacher", "//")]
    for c in CONDITIONS_P4:
        cs = cond_stats(c)
        bars_def.append(([cs[g][0] for g in range(n_groups)],
                         [cs[g][1] for g in range(n_groups)],
                         COND_COLORS[c], COND_LABELS[c], None))
    fig, ax = plt.subplots(figsize=(12, 5))
    for bi, (means, stds, color, label, hatch) in enumerate(bars_def):
        offsets = x + (bi - n_bars/2 + .5) * width
        b = ax.bar(offsets, means, width, yerr=stds, capsize=3, color=color,
                   alpha=.85, label=label)
        if hatch:
            for bar in b:
                bar.set_hatch(hatch)
    ax.set_xticks(x); ax.set_xticklabels(pair_labels)
    ax.set_ylabel("Fisher Discriminant Ratio")
    ax.set_title("FDR on critical pairs — vanilla roughly doubles the teacher's separation;\n"
                 "all SP-KD conditions sit slightly below the teacher and on top of each other",
                 fontsize=11)
    ax.legend(fontsize=9)
    fig.tight_layout(); save_fig(fig, "fig8_fdr_critical_pairs.png")


def fig9_participation_ratio(ext):
    t_global = ext["teacher"]["pr_global"]
    t_pc = ext["teacher"]["pr_per_class"]
    grouped = group_per_student(ext["per_student"])
    g_stats = cond_mean_std(grouped, lambda e: e["pr_global"])
    labels_all = ["Teacher"] + [COND_LABELS[c] for c in CONDITIONS_P4]
    g_means = [t_global] + [g_stats[c][0] for c in CONDITIONS_P4]
    g_errs = [0.0] + [g_stats[c][1] for c in CONDITIONS_P4]
    colors_all = [TEACHER_COLOR] + [COND_COLORS[c] for c in CONDITIONS_P4]

    pc_stats = {c: [(np.mean([e["pr_per_class"][str(ci)] for e in grouped[c]]),
                     np.std([e["pr_per_class"][str(ci)] for e in grouped[c]]))
                    for ci in range(10)] for c in CONDITIONS_P4}
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    ax = axes[0]
    bars = ax.bar(np.arange(len(labels_all)), g_means, yerr=g_errs, capsize=5,
                  color=colors_all, alpha=.85)
    bars[0].set_hatch("//")
    pad = max(g_means) * .01
    for xi, (val, err) in enumerate(zip(g_means, g_errs)):
        ax.text(xi, val + err + pad, f"{val:.2f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(np.arange(len(labels_all))); ax.set_xticklabels(labels_all)
    ax.set_ylabel("Participation Ratio (global)"); ax.axhline(10, color="grey",
                                                              ls=":", lw=1)
    ax.set_title("Global PR — all SP-KD conditions pinned at ≈ n_classes (=10);\n"
                 "vanilla 9.22, teacher 9.33", fontsize=11)
    ax2 = axes[1]; width = .14; n_bars = 5; xpc = np.arange(10)
    pc_bars = [([float(t_pc[str(ci)]) for ci in range(10)], [0.]*10,
                TEACHER_COLOR, "Teacher", "//")]
    for c in CONDITIONS_P4:
        pc_bars.append(([pc_stats[c][ci][0] for ci in range(10)],
                        [pc_stats[c][ci][1] for ci in range(10)],
                        COND_COLORS[c], COND_LABELS[c], None))
    for bi, (means, stds, color, label, hatch) in enumerate(pc_bars):
        offsets = xpc + (bi - n_bars/2 + .5) * width
        b = ax2.bar(offsets, means, width, yerr=stds, capsize=2, color=color,
                    alpha=.85, label=label)
        if hatch:
            for bar in b:
                bar.set_hatch(hatch)
    ax2.set_xticks(xpc); ax2.set_xticklabels(CIFAR10_CLASSES, rotation=45,
                                             ha="right", fontsize=9)
    ax2.set_ylabel("Participation Ratio (per class)"); ax2.set_title(
        "Per-class PR", fontsize=11)
    ax2.legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "fig9_participation_ratio.png")


fig7_intra_class_variance(ARCHIVED["extended"])
fig8_fdr_critical_pairs(ARCHIVED["extended"])
fig9_participation_ratio(ARCHIVED["extended"])

In [ ]:
# ── Tables — behaviour: confidence, critical-pair confusions (§5.5) ────────
conf = ARCHIVED["confidence"]
grouped_conf = group_per_student(conf["per_student"])

rows = {"Teacher": {
    "acc": round(conf["teacher"]["accuracy"], 4),
    "conf (correct)": round(conf["teacher"]["mean_confidence_correct"], 4),
    "entropy": round(conf["teacher"]["mean_entropy_overall"], 4),
    "logit margin": round(conf["teacher"]["mean_logit_margin"], 3),
    "calibration gap": round(conf["teacher"]["calibration_gap"], 4),
}}
for cond in CONDITIONS_P4:
    entries = grouped_conf[cond]
    n_crit = int(round(float(np.mean([
        sum(v["count"] for v in e["critical_pairs"].values()) for e in entries]))))
    dog_as_cat = float(np.mean([e["critical_pairs"]["dog_as_cat"]["confidence"]
                                for e in entries]))
    rows[COND_LABELS[cond]] = {
        "acc": round(float(np.mean([e["accuracy"] for e in entries])), 4),
        "conf (correct)": round(float(np.mean(
            [e["mean_confidence_correct"] for e in entries])), 4),
        "entropy": round(float(np.mean(
            [e["mean_entropy_overall"] for e in entries])), 4),
        "logit margin": round(float(np.mean(
            [e["mean_logit_margin"] for e in entries])), 3),
        "calibration gap": round(float(np.mean(
            [e["calibration_gap"] for e in entries])), 4),
    }
    rows[COND_LABELS[cond]]["critical-pair confusions (6 dirs)"] = n_crit
    rows[COND_LABELS[cond]]["dog→cat conf."] = round(dog_as_cat, 3)

n_teacher_crit = sum(v["count"] for v in conf["teacher"]["critical_pairs"].values())
rows["Teacher"]["critical-pair confusions (6 dirs)"] = n_teacher_crit
rows["Teacher"]["dog→cat conf."] = round(
    conf["teacher"]["critical_pairs"]["dog_as_cat"]["confidence"], 3)

print("Behavioural summary (paper §5.5: teacher 168 critical-pair confusions;"
      " bi_rep commits most among BI conditions and misclassifies dog→cat at 0.883):")
display(pd.DataFrame(rows).T)

In [ ]:
# ── Figures P4-C1 / P4-C6 + robustness tables (§5.6) ───────────────────────
def figc1_mce_bar(c10c):
    agg = c10c["aggregate"]
    means = [agg[c]["mCE_mean"] for c in CONDITIONS_P4]
    stds = [agg[c]["mCE_std"] for c in CONDITIONS_P4]
    x = np.arange(len(CONDITIONS_P4))
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(x, means, yerr=stds, capsize=5, width=.55,
                  color=[COND_COLORS[c] for c in CONDITIONS_P4], alpha=.85)
    ax.axhline(1.0, color=TEACHER_COLOR, ls="--", lw=1.5, label="Teacher (mCE = 1.0)")
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + .005, f"{m:.3f}",
                ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS_P4])
    ax.set_ylabel("Mean Corruption Error (mCE)")
    ax.set_title("CIFAR-10-C: vanilla — least teacher-faithful — is the MOST robust student")
    ax.legend(fontsize=9)
    fig.tight_layout(); save_fig(fig, "figc1_mce_bar.png")


def figc6_ece_comparison(c10c):
    agg = c10c["aggregate"]
    x = np.arange(len(CONDITIONS_P4)); w = .35
    clean_means = [agg[c]["mean_ece_clean"] for c in CONDITIONS_P4]
    clean_stds = [agg[c]["mean_ece_clean_std"] for c in CONDITIONS_P4]
    corr_means = [agg[c]["mean_ece_corrupted"] for c in CONDITIONS_P4]
    corr_stds = [agg[c]["mean_ece_corrupted_std"] for c in CONDITIONS_P4]
    t_clean = c10c["teacher_clean_ece"]
    t_corr = float(np.mean(list(c10c["ece_per_corruption"]["teacher"].values())))
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - w/2, clean_means, w, yerr=clean_stds, capsize=4,
           color=[COND_COLORS[c] for c in CONDITIONS_P4], alpha=.85, label="Clean ECE")
    ax.bar(x + w/2, corr_means, w, yerr=corr_stds, capsize=4,
           color=[COND_COLORS[c] for c in CONDITIONS_P4], alpha=.45, hatch="//",
           label="Corrupted ECE (mean)")
    ax.axhline(t_clean, color=TEACHER_COLOR, lw=1.5,
               label=f"Teacher clean ECE ({t_clean:.3f})")
    ax.axhline(t_corr, color=TEACHER_COLOR, ls="--", lw=1.5,
               label=f"Teacher corrupted ECE ({t_corr:.3f})")
    ax.set_xticks(x); ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS_P4])
    ax.set_ylabel("Expected Calibration Error")
    ax.set_title("Calibration clean vs corruption-averaged", fontsize=11)
    ax.legend(fontsize=8)
    fig.tight_layout(); save_fig(fig, "figc6_ece_comparison.png")


c10c = ARCHIVED["cifar10c"]
figc1_mce_bar(c10c)
figc6_ece_comparison(c10c)

rob_rows = {}
for cond in CONDITIONS_P4:
    a = c10c["aggregate"][cond]
    rob_rows[COND_LABELS[cond]] = {
        "mCE ↓": f'{a["mCE_mean"]:.3f} ± {a["mCE_std"]:.3f}',
        "RmCE": round(a["rmCE_mean"], 3),
        "ECE clean": round(a["mean_ece_clean"], 4),
        "ECE corrupted": round(a["mean_ece_corrupted"], 3),
    }
rob_rows["Teacher"] = {"mCE ↓": "1.000 (ref)", "RmCE": "—",
                       "ECE clean": round(c10c["teacher_clean_ece"], 4),
                       "ECE corrupted": round(float(np.mean(
                           list(c10c["ece_per_corruption"]["teacher"].values()))), 3)}
print("Robustness table (paper §5.6):")
display(pd.DataFrame(rob_rows).T)

# Per-corruption calibration nuance: never worse than the teacher
ece = c10c["ece_per_corruption"]
t_ece = ece["teacher"]
nuance = {}
for cond in CONDITIONS_P4:
    cond_ece = ece[f"cond_{cond}"]
    worse = sum(1 for corr in c10c["corruption_types"]
                if cond_ece[corr] > t_ece[corr])
    nuance[COND_LABELS[cond]] = f"{worse}/19 corruptions worse than teacher"
print("bi_rep-only property (verified per-corruption in paper §5.6):")
display(pd.Series(nuance).rename("ECE > teacher count").to_frame().T)

p10p = ARCHIVED["cifar10p"]
stab_rows = {COND_LABELS[c]: {
    "mFP": f'{p10p["aggregate"][c]["mFP_mean"]:.4f} ± {p10p["aggregate"][c]["mFP_std"]:.4f}',
    "mFPR": round(p10p["aggregate"][c]["mFPR_mean"], 3)} for c in CONDITIONS_P4}
stab_rows["Teacher"] = {"mFP": round(p10p["teacher_mFP"], 4), "mFPR": "—"}
print("CIFAR-10-P stability (all students more stable than the teacher; no separation):")
display(pd.DataFrame(stab_rows).T)

### 5.3 · Optional: recompute Phase-4 statistics from student checkpoints

If the original student checkpoints (`{condition}_student_seed_{seed}.pt`,
ResNet-18 CIFAR-stem) are placed in `/content/phase4_results/checkpoints`
(e.g. uploaded or mounted from Google Drive), this cell recomputes the fidelity /
geometry / behaviour statistics live and regenerates the figures from measured
values instead of the archive. Empty directory ⇒ silently skipped.

In [ ]:
STUDENT_CKPT_DIR = PHASE4_DIR / "checkpoints"

@torch.no_grad()
def inference_stats(model, loader, device=DEVICE):
    logits = torch.cat([model(img.to(device)).cpu() for img, _ in loader])
    targets = torch.cat([lbl for _, lbl in loader])
    probs = F.softmax(logits, dim=1)
    preds = logits.argmax(dim=1)
    N = len(targets)
    correct = preds == targets
    true_probs = probs[torch.arange(N), targets]
    crit = {}
    for name, (tc, pc_) in {"cat_as_dog": (3, 5), "dog_as_cat": (5, 3),
                            "cat_as_deer": (3, 4), "deer_as_cat": (4, 3),
                            "dog_as_deer": (5, 4), "deer_as_dog": (4, 5)}.items():
        mask = (targets == tc) & (preds == pc_)
        crit[name] = {"confidence": float(probs[mask].max(dim=1).values.mean()) if mask.any() else 0.,
                      "count": int(mask.sum())}
    other = logits.clone(); other[torch.arange(N), targets] = -float("inf")
    margin = logits[torch.arange(N), targets] - other.max(dim=1).values
    p_clamped = probs.clamp(min=1e-12)
    entropy = -(p_clamped * p_clamped.log()).sum(dim=1)
    return {"accuracy": round(correct.float().mean().item(), 6),
            "mean_confidence_correct": round(true_probs[correct].mean().item(), 6),
            "mean_entropy_overall": round(entropy.mean().item(), 6),
            "mean_logit_margin": round(margin[correct].mean().item(), 6) if correct.any() else 0.,
            "calibration_gap": round(true_probs[correct].mean().item()
                                     - correct.float().mean().item(), 6),
            "critical_pairs": crit}


def _pr_single(mat):
    if mat.shape[0] <= 1:
        return 1.0
    mat_c = mat - mat.mean(dim=0, keepdim=True)
    sv2 = torch.linalg.svdvals(mat_c) ** 2
    denom = (sv2 ** 2).sum()
    return float((sv2.sum() ** 2 / denom).item()) if denom > 1e-12 else 1.0


def geometry_metrics(F_mat, labels):
    centroids, variances = [], []
    for k in range(10):
        F_k = F_mat[labels == k]
        mu_k = F_k.mean(dim=0)
        centroids.append(mu_k)
        variances.append(float(((F_k - mu_k) ** 2).sum(dim=1).mean()))
    icv_mean = float(np.mean(variances))
    fdr = {}
    for name, (a, b) in CRITICAL_PAIRS.items():
        d2 = float(((centroids[a] - centroids[b]) ** 2).sum())
        fdr[name] = d2 / (variances[a] + variances[b])
    mean_fdr = float(np.mean([
        float(((centroids[a] - centroids[b]) ** 2).sum()) / (variances[a] + variances[b])
        for a in range(10) for b in range(a + 1, 10)]))
    pr_pc = {str(k): round(_pr_single(F_mat[labels == k]), 4) for k in range(10)}
    return {"icv_mean": round(icv_mean, 6), "fdr_critical_pairs": fdr,
            "fdr_mean_all_pairs": round(mean_fdr, 6),
            "pr_global": round(_pr_single(F_mat), 4), "pr_per_class": pr_pc}


def recompute_from_checkpoints():
    ckpts = sorted(STUDENT_CKPT_DIR.glob("*_student_seed_*.pt"))
    if not ckpts:
        print(f"No checkpoints in {STUDENT_CKPT_DIR} — using embedded archive (nothing done).")
        return None
    print(f"Found {len(ckpts)} checkpoint(s) — recomputing …")
    fine_ps, ext_ps, conf_ps = {}, {}, {}
    F_teacher, labels_cal = extract_features_with_labels(teacher, calib_loader, DEVICE)
    S_t = class_cosine_matrix(F_teacher.to(DEVICE), labels_cal.to(DEVICE))
    t_geom = geometry_metrics(F_teacher.to(DEVICE), labels_cal.to(DEVICE))
    t_stats = inference_stats(teacher, test_loader, DEVICE)
    for path in tqdm(ckpts):
        m = re.match(r"^([a-z]+)_student_seed_(\d+)\.pt$", path.name)
        cond_raw, seed = m.group(1), m.group(2)
        cond = COND_NORM.get(cond_raw, cond_raw)
        key = f"{cond_raw}_seed_{seed}"
        student_m = ResNet18_CIFAR(num_classes=10).to(DEVICE)
        student_m.load_state_dict(torch.load(path, map_location=DEVICE))
        student_m.eval()
        F_s, _ = extract_features_with_labels(student_m, calib_loader, DEVICE)
        S_s = class_cosine_matrix(F_s.to(DEVICE), labels_cal.to(DEVICE))
        pcc = {str(k): round(linear_cka(F_teacher[labels_cal == k], F_s[labels_cal == k]), 6)
               for k in range(10)}
        fine_ps[key] = {
            "cosine_diff_frob": round(float(torch.norm(S_s - S_t, p="fro").item()), 6),
            "class_cka_variance": round(float(np.var(list(pcc.values()))), 8)}
        ext_ps[key] = geometry_metrics(F_s.to(DEVICE), labels_cal.to(DEVICE))
        conf_ps[key] = inference_stats(student_m, test_loader, DEVICE)
        del student_m
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    live = {"fine": {"per_student": fine_ps},
            "extended": {"teacher": t_geom, "per_student": ext_ps},
            "confidence": {"teacher": t_stats, "per_student": conf_ps}}
    with open(PHASE4_DIR / "phase4_live_recompute.json", "w") as f:
        json.dump(live, f, indent=2)
    print("Done → phase4_results/phase4_live_recompute.json\n"
          "Regenerating figures from MEASURED values:")
    fig1_aggregate_bar(live["fine"]["per_student"], title_suffix="(live recompute)")
    fig7_intra_class_variance(live["extended"])
    fig8_fdr_critical_pairs(live["extended"])
    fig9_participation_ratio(live["extended"])
    return live

live_recompute = recompute_from_checkpoints()

### 5.4 · Optional: live CIFAR-10-C evaluation

Faithful port of the robustness evaluation: per-corruption accuracy/ECE over 19
corruptions × 5 severities, teacher-normalised **mCE**
($\mathrm{CE}_c=\sum_s(1-\mathrm{acc}_c)/\sum_s(1-\mathrm{acc}^{T}_c)$) and
**RmCE** ($\sum_s(\mathrm{acc}_{clean}-\mathrm{acc}_{c,s}) / \sum_s(\mathrm{acc}^T_{clean}-\mathrm{acc}^T_{c,s})$),
15-bin ECE (clean + corrupted). Downloads CIFAR-10-C (~2.9 GB) from Zenodo.
`RUN_CIFAR10C_EVAL=False` by default — the reported values are regenerated from
the archive above; CIFAR-10-P flip-probability numbers likewise come from the
archive (`cifar10p` key).

In [ ]:
CIFAR10C_URL = "https://zenodo.org/records/2535967/files/CIFAR-10-C.tar.gz"
CIFAR10P_URL = "https://zenodo.org/records/2535967/files/CIFAR-10-P.tar.gz"
CORRUPTION_TYPES = [
    "gaussian_noise", "shot_noise", "impulse_noise", "defocus_blur",
    "glass_blur", "motion_blur", "zoom_blur", "snow", "frost", "fog",
    "brightness", "contrast", "elastic_transform", "pixelate",
    "jpeg_compression", "speckle_noise", "gaussian_blur", "spatter", "saturate"]

def preprocess_numpy_images(images_np):
    mean = torch.tensor(CIFAR10_MEAN).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(1, 3, 1, 1)
    t = torch.from_numpy(images_np.astype(np.float32)).div(255.0).permute(0, 3, 1, 2)
    return (t - mean) / std


def compute_ece(confidences, correct, n_bins=15):
    edges = torch.linspace(0., 1., n_bins + 1)
    ece = 0.
    for i in range(n_bins):
        mask = ((confidences >= edges[i]) & (confidences <= edges[i + 1])) \
               if i == n_bins - 1 else \
               ((confidences >= edges[i]) & (confidences < edges[i + 1]))
        if mask.sum() == 0:
            continue
        ece += (mask.float().mean().item()) * abs(
            correct[mask].float().mean().item() - confidences[mask].mean().item())
    return float(ece)


@torch.no_grad()
def eval_on_images(model, images_np, labels_np, device=DEVICE):
    probs_all = []
    ds = TensorDataset(preprocess_numpy_images(images_np),
                       torch.from_numpy(labels_np.astype(np.int64)))
    loader = DataLoader(ds, batch_size=BATCH_SIZE_INFERENCE, shuffle=False)
    for imgs, _ in loader:
        probs_all.append(torch.softmax(model(imgs.to(device)), dim=1).cpu())
    probs = torch.cat(probs_all)
    preds = probs.argmax(dim=1)
    confs = probs.max(dim=1).values
    correct = preds == torch.from_numpy(labels_np.astype(np.int64))
    return {"accuracy": correct.float().mean().item(),
            "ece": compute_ece(confs, correct)}


def download_cifar_c_or_p(url, dest_dir):
    if dest_dir.exists() and (dest_dir / "labels.npy").exists():
        return
    archive = dest_dir.parent / Path(url).name
    print(f"Downloading {url} …")
    urllib.request.urlretrieve(url, archive)
    with tarfile.open(archive) as tf:
        tf.extractall(dest_dir.parent)
    archive.unlink()


def run_cifar10c_eval(models_to_eval):
    dest = DATA_DIR / "CIFAR-10-C"
    download_cifar_c_or_p(CIFAR10C_URL, dest)
    labels = np.load(dest / "labels.npy")
    clean_labels = np.array(test_ds.targets)

    def evaluate_model(model):
        clean_r = eval_on_images(model,
                                 test_ds.data, clean_labels, DEVICE)
        per_corr = {}
        for corr in CORRUPTION_TYPES:
            accs, eces = [], []
            for sev in range(1, 6):
                imgs = np.load(dest / f"{corr}.npy")[(sev - 1) * 10000: sev * 10000]
                r = eval_on_images(model, imgs, labels, DEVICE)
                accs.append(r["accuracy"]); eces.append(r["ece"])
            per_corr[corr] = {"accuracies": accs, "eces": eces}
        return {"clean_accuracy": clean_r["accuracy"], "clean_ece": clean_r["ece"],
                "per_corruption": per_corr}

    print("Evaluating teacher …")
    results = {name: evaluate_model(m) for name, m in models_to_eval.items()}
    t_res = results["teacher"]
    t_pc = {c: [1.0 - a for a in t_res["per_corruption"][c]["accuracies"]]
            for c in CORRUPTION_TYPES}

    def compute_mce(pc):
        ces = []
        for c in CORRUPTION_TYPES:
            model_err = [1.0 - a for a in pc[c]["accuracies"]]
            denom = sum(t_pc[c])
            ces.append(sum(model_err) / denom if denom > 0 else 0.0)
        return float(np.mean(ces))

    def compute_rmce(pc, clean_acc):
        rmces = []
        for c in CORRUPTION_TYPES:
            drops = [clean_acc - a for a in pc[c]["accuracies"]]
            denom = sum([t_res["clean_accuracy"] - a
                         for a in t_res["per_corruption"][c]["accuracies"]])
            rmces.append(sum(drops) / denom if denom > 0 else 0.0)
        return float(np.mean(rmces))

    summary = {}
    for name, res in results.items():
        summary[name] = {
            "mCE": round(compute_mce(res["per_corruption"]), 4),
            "RmCE": round(compute_rmce(res["per_corruption"],
                                       res["clean_accuracy"]), 4),
            "mean_corr_ECE": round(float(np.mean(
                [np.mean(res["per_corruption"][c]["eces"])
                 for c in CORRUPTION_TYPES])), 4),
            "clean_ece": round(res["clean_ece"], 4)}
    display(pd.DataFrame(summary).T)
    with open(PHASE4_DIR / "phase4_cifar10c_live.json", "w") as f:
        json.dump({"per_model": results, "summary": summary}, f, indent=2)
    return results

if RUN_CIFAR10C_EVAL:
    models_to_eval = {"teacher": teacher}
    ckpts = sorted(STUDENT_CKPT_DIR.glob("*_student_seed_*.pt")) if STUDENT_CKPT_DIR.exists() else []
    for p in ckpts:
        m = re.match(r"^([a-z]+)_student_seed_(\d+)\.pt$", p.name)
        net = ResNet18_CIFAR(num_classes=10).to(DEVICE)
        net.load_state_dict(torch.load(p, map_location=DEVICE)); net.eval()
        models_to_eval[p.stem] = net
    cifar10c_live = run_cifar10c_eval(models_to_eval)
else:
    print("RUN_CIFAR10C_EVAL = False — reported robustness numbers regenerated from "
          "the archived snapshot above.")

<a id="s6"></a>
## 6 · Artifact map & export

Every figure and table of the report is produced above:

| Report element | Notebook location |
|---|---|
| Table 1 (16-block BI metrics) | §2 output table (`phase2_results.json`) |
| Fig. P3-1/2/3 (τ matrix, grouped bars, Jaccard) | §3 `fig1_tau_heatmap`, `fig2_grouped_bar`, `fig3_jaccard_heatmaps` |
| Fig. P3-4 (silent-failure scatter) | §3 `fig4_scatter` |
| Fig. P3-6/7 (per-class CKA, entropy shift) | §3 `fig6_*`, `fig7_entropy_layer4_0` |
| Fig. P3-10/11 (class heatmaps, pair mergers) | §3 `fig10_*`, `fig11_class_pair_changes_layer4_0` |
| Fig. P3-8/9 (Gram cross-check, class-level bars) | §3 `fig8_birep_vs_gram`, `fig9_birep_class_bar` |
| Fig. P3-13/14 (per-class heatmap, top-5) | §3 `fig13_*`, `fig14_*` |
| Fig. P3-12/15 (simulated & real pruning) | §3 `fig12_progressive_pruning`; §4 `fig_progressive_pruning_real` + superadditivity table |
| Weight-vector collapse & γ diagnosis (Causes 1–2) | §5 live computations + `fig_sp_loss_share` |
| Final accuracy/CKA table + Welch tests | §5.2 |
| Fig. P4-1 (aggregate fidelity) | §5.2 `fig1_aggregate_bar` |
| Fig. P4-7/8/9 (ICV, FDR, PR) | §5.2 `fig7_intra_class_variance`, `fig8_fdr_critical_pairs`, `fig9_participation_ratio` |
| Behavioural table (critical-pair confusions) | §5.2 |
| Fig. P4-C1/C6 (mCE, ECE) + robustness tables | §5.2 `figc1_mce_bar`, `figc6_ece_comparison` |

References: Tung & Mori (2019); Hinton et al. (2015); Kornblith et al. (2019);
Ding et al. (2021); Men et al. (2024); Hendrycks & Dietterich (2019) — full list
in the report.

In [ ]:
# ── Export everything produced in this session ─────────────────────────────
import shutil

stamp = time.strftime("%Y%m%d_%H%M%S")
zip_base = f"/content/block_influence_artifacts_{stamp}"
tmp = Path("/content/_export")
if tmp.exists():
    shutil.rmtree(tmp)
tmp.mkdir()
shutil.copytree(RESULTS_DIR, tmp / "results",
                ignore=shutil.ignore_patterns("*.pt"))
shutil.copytree(FIGURES_DIR, tmp / "figures")
shutil.make_archive(zip_base, "zip", tmp)
print(f"Artifacts bundled → {zip_base}.zip "
      f"({Path(zip_base + '.zip').stat().st_size/1e6:.1f} MB)\nContents:")
for p in sorted(tmp.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(tmp))

try:
    from google.colab import files
    files.download(f"{zip_base}.zip")
except Exception as e:
    print("(download the zip manually from the file browser:", e, ")")

---
*Notebook generated from `Bi_project/code/*` — pipeline ported verbatim except
for the documented deviations in the header. All reported numbers trace to the
result artifacts listed in Appendix A.2 of the report.*